# Aquatic Guardian — Mentee 1 (Workstream A)
## Aquatic Monitoring: Detection, Event Evidence, Ecological Inference, and Edge Deployment

**Six-week Colab project. Free tier, open-source only, public data.**

---

### What this notebook is

A **monitoring system**, evaluated as a monitoring system. Not a classifier with an accuracy
number attached.

The charter asks four questions that a confusion matrix cannot answer:

1. *Can an edge device reliably detect selected aquatic organisms and ecological events
   under realistic conditions?* — realistic means turbid, dark, and at range, so the
   detector has to be scored **against the optical physics that limits it**.
2. *Which observations should trigger local retention, operator notification, or cloud
   upload?* — that is a **decision problem with asymmetric costs**, not a threshold.
3. *How much video can be dropped without losing scientifically important events?* — a
   **bitrate budget** with a measured recall cost.
4. *Can video detections improve the interpretation of temperature, DO, turbidity and
   salinity data?* — a **fusion and inference** problem, and the place where an imperfect
   detector quietly corrupts the ecology if nobody corrects for it.

```
  optics & range budget ──▶ synthetic scenes with exact GT ──▶ detector (baseline + deep)
   (Jaffe-McGlamery,          (range, turbidity, light,           │
    Jerlov water types)        occlusion all controlled)          ▼
                                                        evaluation: AP, PR, per-condition,
                                                        bootstrap CIs, calibration (ECE)
                                                                  │
          ┌───────────────────────────────────────────────────────┤
          ▼                                                       ▼
   conformal abstention                              cost-optimal operating point
   (guaranteed coverage)                             + base-rate / alert-budget maths
          │                                                       │
          └──────────────────▶ tracking (Kalman + Hungarian) ◀────┘
                                       │  MOTA / IDF1 / HOTA
                                       ▼
              ecological inference: occupancy model correcting for imperfect detection
                                       │
                                       ▼
             sensor fusion with the thermal notebook  ──▶  edge deployment + data reduction
                                       │                    (INT8, latency, energy, bitrate)
                                       ▼
                        drift detection, safety rules, handoff
```

### How this differs from Mentee 2's vision work

| | **Mentee 1 (this notebook)** | **Mentee 2** |
|---|---|---|
| Where | **outside** the facility, in the receiving water | **inside** a contained aquaculture loop |
| Task | detection, classification, event evidence | instance segmentation and measurement |
| Output | alerts, retention decisions, occupancy/abundance | length, mass, biomass, welfare score |
| Scored on | AP, calibration, alert precision, MOT, detection probability | length/mass MAPE, IoU, welfare agreement |
| Consumes | the thermal notebook's plume and water-quality fields | the thermal notebook's loop inlet temperature |

### Scope reminder

A detection is **evidence**, never a verdict. This notebook produces the evidence that lets a
human decide; it does not decide. Specifically: **a detection is never, on its own, evidence
that an ecosystem is healthy or unsafe.** Degraded images, unseen organisms and low-confidence
cases are routed to `unknown` and escalated, never silently absorbed into a class.

---

### Six-week map

| Week | Parts | Deliverable |
|---|---|---|
| 1 | 0–2 | Dataset + licence + split discipline; optical range budget for the site |
| 2 | 3–5 | Scene generator with exact GT; detector; evaluation harness **passing its vetting suite** |
| 3 | 6–7 | Calibration, conformal abstention, cost-optimal operating point, alert budget |
| 4 | 8–9 | Tracking + MOT metrics; occupancy model correcting for imperfect detection |
| 5 | 10–11 | Sensor fusion with the thermal model; edge deployment + data-reduction analysis |
| 6 | 12–14 | Drift detection, agricultural transfer, error taxonomy, reproducible handoff |

## Part 0 — Environment, run configuration, and the safety contract

In [ ]:
# ---- library install (Colab vs local) ----------------------------------------
# Local / CI: editable install from the repo parent of notebooks/.
# Colab (optional Drive path for handoffs): /content/drive/MyDrive/aquatic_guardian/
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "-q", "install",
        "git+https://github.com/ejri/aquatic-guardian.git@v0.1.0",
    ])
else:
    _repo = Path("..").resolve()
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "-e", str(_repo)])

import aquatic_guardian as ag
print("aquatic_guardian", ag.__version__)

HANDOFF_DIR = Path("_handoffs")
HANDOFF_DIR.mkdir(parents=True, exist_ok=True)
# Colab optional: HANDOFF_DIR = Path("/content/drive/MyDrive/aquatic_guardian")
print("handoff directory:", HANDOFF_DIR.resolve())


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json, time, os, textwrap
from scipy import ndimage as ndi, stats
from scipy.optimize import linear_sum_assignment, minimize

np.set_printoptions(suppress=True, precision=4)
pd.set_option("display.width", 165); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 9, "figure.facecolor": "white"})

RUN = dict(
    SEED          = 20260219,
    RUN_REAL_DATA = False,   # True in Colab: download a public dataset + train the CNN
    FAST          = True,    # False = more frames, more bootstrap draws (slower)
    EXPORT_DIR    = ".",
)
RNG = np.random.default_rng(RUN["SEED"])

SAFETY_CONTRACT = {
  "role_of_this_system": "evidence generation and triage for human review",
  "never": ["declare an ecosystem healthy or unsafe",
            "act autonomously on an alert",
            "treat a degraded frame as a confident negative",
            "report abundance without correcting for detection probability"],
  "abstention": "low-confidence and out-of-distribution frames route to 'unknown' and escalate",
  "fail_safe": "missing, stale or out-of-range input escalates; it never reads as 'nothing there'",
  "human_review_required_for": ["any ecological claim", "any retention or deletion policy "
                                "put into service", "any alert threshold change"],
}
print("Aquatic Guardian — Mentee 1 — aquatic monitoring stack")
print(f"seed={RUN['SEED']}  real_data={RUN['RUN_REAL_DATA']}  fast={RUN['FAST']}")
for k, v in SAFETY_CONTRACT.items():
    print(f"  {k}: {v}")

## Part 1 — Dataset, licence, and the discipline that decides whether your number is real

### 1.1 Candidate public datasets

Record the licence **in the repo, before training**. The licence decides whether you may
redistribute the data, the derived weights, or neither.

In [ ]:
DATASETS = pd.DataFrame([
 ("DeepFish", "~40,000 images", "reef, tropical Australia, mostly clear",
  "classification, counting, segmentation subsets", "MIT / CC BY 4.0",
  "github.com/alzayats/DeepFish"),
 ("Brackish", "~14,700 frames, 89 videos", "brackish strait, HIGHLY variable visibility",
  "6-class object detection (fish, crab, jellyfish, shrimp, starfish, small fish)",
  "CC BY 4.0", "public.roboflow.com/object-detection/brackish-underwater"),
 ("Fish4Knowledge", "video + extracted frames", "reef, 10-year monitoring, real fouling",
  "detection, tracking, species", "check per-clip terms",
  "homepages.inf.ed.ac.uk/rbf/Fish4Knowledge"),
 ("OzFish", "~80,000 crops from BRUVS", "Australian coastal, baited remote video",
  "detection + species", "CC BY 4.0", "github.com/open-AIMS/ozfish"),
 ("FishNet / NOAA AFSC", "survey imagery", "NE Pacific, cold water — closest to a BC site",
  "detection + species", "US Gov public domain (verify)", "fishnet.ai ; NOAA AFSC repositories"),
], columns=["dataset", "size", "conditions", "task", "licence", "source"])
display(DATASETS)

print("Choose on CONDITION MATCH, not size. A 40k-image clear-water reef dataset will")
print("train a detector that fails on a turbid BC winter morning. Brackish and Fish4Knowledge")
print("carry the visibility variation that the deployment actually has; Part 2 makes that")
print("statement quantitative instead of a hunch.")

def download_real_dataset(which="Brackish"):
    '''Guarded. Records provenance so the repo can reproduce the split without the raw data.'''
    import subprocess
    if which == "DeepFish":
        if not os.path.exists("DeepFish"):
            subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/alzayats/DeepFish.git"], check=True)
    else:
        print("Download from the source column above (Roboflow API or the paper's data link).")
    print("Write dataset name, version/commit, licence and retrieval date to data_provenance.json.")

if RUN["RUN_REAL_DATA"]:
    download_real_dataset("Brackish")
else:
    print("\nReal download skipped. The physically-based generator in Part 3 stands in, and")
    print("gives exact ground truth for range, turbidity and occlusion that no public dataset has.")

### 1.2 Split discipline — the single most common way an underwater CV result becomes fiction

Frames from one video clip are **near-duplicates**. Splitting randomly by frame puts almost
identical images in train and test, and the reported accuracy measures memorisation, not
generalisation. The rule is: **split by clip, session, day and deployment — never by frame.**

The cell below demonstrates the size of the lie, using data where we know the truth.

In [ ]:
def leakage_demo(n_clips=40, frames_per_clip=25, clip_effect=2.2, noise=0.7, seed=0):
    '''Each clip has its own strong nuisance signature (lighting, fouling, camera pose).
    A model can score well by recognising the CLIP rather than the CLASS.'''
    r = np.random.default_rng(seed)
    clip_sig = r.normal(0, clip_effect, (n_clips, 4))
    y_clip = r.integers(0, 2, n_clips)
    X, y, clip = [], [], []
    for c in range(n_clips):
        for _ in range(frames_per_clip):
            f = clip_sig[c] + r.normal(0, noise, 4)
            f[0] += 1.0*y_clip[c]                     # the only real class signal, and it is weak
            X.append(f); y.append(y_clip[c]); clip.append(c)
    return np.array(X), np.array(y), np.array(clip)

def logreg(X, y, epochs=800, lr=.3, l2=1e-3):
    Xb = np.c_[X, np.ones(len(X))]; w = np.zeros(Xb.shape[1])
    for _ in range(epochs):
        p = 1/(1+np.exp(-Xb@w))
        w -= lr*(Xb.T@(p-y)/len(y) + l2*w)
    return w
def acc(X, y, w):
    return float((( 1/(1+np.exp(-(np.c_[X, np.ones(len(X))]@w))) > .5).astype(int) == y).mean())

X, y, clip = leakage_demo()
rs = np.random.default_rng(1).permutation(len(X)); cut = int(.7*len(X))
tr_f, te_f = rs[:cut], rs[cut:]                                     # WRONG: random by frame
test_clips = np.arange(40)[np.random.default_rng(1).permutation(40)][:12]
te_c = np.isin(clip, test_clips); tr_c = ~te_c                      # RIGHT: grouped by clip

w_f = logreg(X[tr_f], y[tr_f]); w_c = logreg(X[tr_c], y[tr_c])
res = pd.DataFrame([
  dict(split="random by FRAME (leaky)", train_acc=acc(X[tr_f], y[tr_f], w_f),
       test_acc=acc(X[te_f], y[te_f], w_f)),
  dict(split="grouped by CLIP (honest)", train_acc=acc(X[tr_c], y[tr_c], w_c),
       test_acc=acc(X[te_c], y[te_c], w_c)),
]).set_index("split").round(3)
display(res)
gap = res.test_acc.iloc[0] - res.test_acc.iloc[1]
print(f"Leakage inflates reported test accuracy by {100*gap:.0f} percentage points on data where")
print("the class signal is deliberately weak. Every underwater dataset has clip signatures this")
print("strong or stronger. Report the grouped number, and say in the README how you grouped.")
print("\nRule for this project: group by (deployment, day, clip). If a frame's neighbours could")
print("be in the other split, the split is wrong.")

### 1.3 Annotation agreement and how many frames you actually need

Two things nobody budgets for and both sink projects.

**Agreement.** If two annotators disagree, the ceiling on any model is the agreement, not
100 %. Cohen's κ corrects raw agreement for chance:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

**Sample size.** To report recall with a confidence interval of half-width $w$ you need
positives, and the count follows from the Wilson interval. Most underwater projects
label thousands of *frames* and still have only dozens of *event* examples, which is
why the event class gets a meaningless recall estimate.

In [ ]:
def cohen_kappa(a, b, K):
    n = len(a); M = np.zeros((K, K))
    for i, j in zip(a, b): M[i, j] += 1
    po = np.trace(M)/n
    pe = float((M.sum(0)/n) @ (M.sum(1)/n))
    return (po-pe)/(1-pe+1e-12), po

r = np.random.default_rng(4); n = 600
truth = r.choice([0, 1, 2], n, p=[.55, .30, .15])
def annot(t, skill):
    out = t.copy(); flip = r.random(n) > skill
    out[flip] = r.choice([0, 1, 2], flip.sum())
    return out
rows = []
for skill in [0.98, 0.90, 0.80, 0.70]:
    a, b = annot(truth, skill), annot(truth, skill)
    k, po = cohen_kappa(a, b, 3)
    rows.append(dict(per_annotator_skill=skill, raw_agreement=round(po, 3),
                     cohen_kappa=round(k, 3),
                     max_achievable_model_acc=round(float((a == truth).mean()), 3)))
display(pd.DataFrame(rows).set_index("per_annotator_skill"))
print("kappa < 0.6 means the label definition is ambiguous, not that the annotators are bad.")
print("Fix the annotation guide before labelling more frames; relabelling is far cheaper than")
print("discovering later that the ceiling was 0.8 all along.\n")

def wilson_halfwidth(p, n, z=1.96):
    d = 1 + z*z/n
    c = p + z*z/(2*n)
    hw = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return hw/d

print("Positives needed for a given confidence-interval half-width on recall (95% Wilson):")
tab = []
for p in [0.70, 0.85, 0.95]:
    row = dict(true_recall=p)
    for w in [0.10, 0.05, 0.02]:
        n = next((n for n in range(5, 20001) if wilson_halfwidth(p, n) <= w), None)
        row[f"n_for_+-{w:.2f}"] = n
    tab.append(row)
display(pd.DataFrame(tab).set_index("true_recall"))
print("A rare event class with 40 labelled positives supports a recall CI of roughly +/-0.15.")
print("Report that interval. A point estimate of 0.85 on 40 examples is not a measurement.")

## Part 2 — The optics that decide what is detectable

### 2.1 The image-formation model

Underwater imaging is not photography with a colour cast. Light is **absorbed** and
**scattered**, and the two act differently. The revised Jaffe–McGlamery model
(Akkaynak & Treibitz) separates them:

$$U_\lambda(x) \;=\; \underbrace{I_\lambda(x)\,e^{-\beta^{D}_\lambda z}}_{\text{direct signal}}
 \;+\; \underbrace{B^{\infty}_\lambda\left(1-e^{-\beta^{B}_\lambda z}\right)}_{\text{backscatter veil}}$$

where $z$ is the range to the object, $\beta^{D}_\lambda$ is the attenuation of the direct
signal, $\beta^{B}_\lambda$ the (different) attenuation governing backscatter, and
$B^{\infty}_\lambda$ the veiling light. The coefficients depend on wavelength — red dies
first — and on water type, catalogued as **Jerlov water bodies**.

### 2.2 The result that should drive every design decision downstream

Take an object of radiance $I_{fg}$ against a background $I_{bg}$ at the same range. The
observed contrast is

$$U_{fg} - U_{bg} \;=\; (I_{fg}-I_{bg})\,e^{-\beta^{D}_\lambda z}$$

**The backscatter term cancels exactly.** Veiling light adds the same offset to object and
background, so it destroys *absolute brightness* information but leaves *contrast* decaying
purely exponentially with range. Two consequences:

- A detector keyed to absolute intensity fails as soon as the veil rises. A
  **contrast-normalised** feature survives much further. This is not a heuristic — it is what
  the model says, and Part 4 builds the features accordingly.
- Detection range has a hard exponential ceiling set by $\beta^{D}$. No amount of model
  capacity buys range once contrast falls below sensor noise. **That is a siting and
  lighting problem, not a machine-learning problem**, and saying so early saves the project
  from spending six weeks trying to train past physics.

### Library note — underwater image formation

`ag.imaging.forward` / `ag.imaging.invert` implement Akkaynak & Treibitz (2018, 2019) with separate `beta_D` and `beta_B`. Frames with transmittance below `t_min` are flagged `recoverable=False` and should lower detection probability rather than count as absences.

In [ ]:
# INTEGRATED: aquatic_guardian library
# Jerlov-like beam attenuation (1/m) for R,G,B, and veiling light colour.
WATER_TYPES = {
  "oceanic I (clear)":   dict(bD=[0.32, 0.055, 0.045], bB=[0.25, 0.05, 0.04],  Binf=[0.02, 0.18, 0.35]),
  "coastal 1 (typical)": dict(bD=[0.62, 0.20,  0.28],  bB=[0.45, 0.18, 0.24],  Binf=[0.12, 0.42, 0.47]),
  "coastal 5 (turbid)":  dict(bD=[1.40, 0.75,  1.05],  bB=[1.05, 0.70, 0.95],  Binf=[0.22, 0.45, 0.42]),
  "estuarine (BC wet)":  dict(bD=[2.10, 1.35,  1.80],  bB=[1.60, 1.25, 1.65],  Binf=[0.30, 0.40, 0.36]),
}

def uw_forward(I, z, wt):
    w = WATER_TYPES[wt]
    bD, bB, Binf = np.array(w["bD"]), np.array(w["bB"]), np.array(w["Binf"])
    return ag.imaging.forward(I, z, bD, bB, Binf)

def uw_inverse(U, z, wt, t_min=1e-30):
    w = WATER_TYPES[wt]
    bD, bB, Binf = np.array(w["bD"]), np.array(w["bB"]), np.array(w["Binf"])
    r = ag.imaging.invert(U, z, bD, bB, Binf, t_min=t_min)
    return r["J"]

def uw_invert_full(U, z, wt):
    """Full library invert including recoverable flag for detection covariates."""
    w = WATER_TYPES[wt]
    return ag.imaging.invert(U, z, np.array(w["bD"]), np.array(w["bB"]), np.array(w["Binf"]))

# ---- VET 1: forward/inverse round trip, AND its conditioning ----------------
# Recovering I from U means dividing by the transmittance exp(-beta_D z). When that is
# tiny the division amplifies floating-point error -- so the round trip is only exact
# where the direct signal still exists. Measuring that limit is more useful than a
# blanket tolerance, because it is the same limit that defeats real dehazing.
_I = np.array([0.72, 0.58, 0.31])
_rows = []
for wt in WATER_TYPES:
    for z in [0.5, 2.0, 5.0, 8.0]:
        _R = uw_inverse(uw_forward(_I, z, wt), z, wt)
        _t = float(np.exp(-np.array(WATER_TYPES[wt]["bD"])*z).min())
        _rows.append(dict(water_type=wt, z_m=z, min_transmittance=_t,
                          round_trip_rel_err=float(np.abs(_R/_I - 1).max()),
                          amplification=1/_t))
_cond = pd.DataFrame(_rows)
_well = _cond[_cond.min_transmittance > 1e-4]
assert _well.round_trip_rel_err.max() < 1e-9, "image-formation model is not self-consistent"
print(f"[VET] where the direct signal survives (transmittance > 1e-4) the forward/inverse")
print(f"      round trip is exact to {_well.round_trip_rel_err.max():.1e} -> PASS")
display(_cond.round(12).pivot(index="z_m", columns="water_type",
                              values="round_trip_rel_err")
        .style.format("{:.2e}").set_caption(
        "Round-trip relative error: exact until the transmittance collapses"))
_worst = _cond.loc[_cond.round_trip_rel_err.idxmax()]
print(f"Worst case: {_worst.water_type} at {_worst.z_m:.0f} m has transmittance "
      f"{_worst.min_transmittance:.1e}, so inversion amplifies rounding "
      f"{_worst.amplification:,.0f}x and the round trip degrades to "
      f"{_worst.round_trip_rel_err:.1e}.")
print("That is not a bug, it is the physics: once the direct signal is gone there is nothing")
print("to recover, and any restoration method is amplifying noise. It is the same wall the")
print("detection-range budget below runs into, arrived at from the other direction.")

# ---- VET 2: contrast decays as exp(-bD z), independent of the veil ----------
Ifg, Ibg = np.array([0.72, 0.58, 0.31]), np.array([0.25, 0.33, 0.30])
worst = 0.0
for wt in WATER_TYPES:
    for z in [0.5, 1, 2, 4, 8]:
        emp = np.abs(uw_forward(Ifg, z, wt)-uw_forward(Ibg, z, wt))/np.abs(Ifg-Ibg)
        thy = np.exp(-np.array(WATER_TYPES[wt]["bD"])*z)
        worst = max(worst, np.abs(emp-thy).max())
assert worst < 1e-12, "contrast does not follow exp(-bD z) — check the model"
print(f"[VET] contrast ratio matches exp(-beta_D z) to {worst:.1e}; the veil cancels -> PASS")

SENSOR_NOISE = 0.012          # per-channel RMS noise of an 8-bit camera at moderate gain
MIN_CONTRAST = 3*SENSOR_NOISE # 3-sigma detectability floor

def detection_range(wt, dI=None, min_contrast=MIN_CONTRAST):
    '''Range at which object-background contrast falls to the sensor-noise floor,
    per channel and for the best channel.'''
    dI = np.abs(Ifg-Ibg) if dI is None else np.asarray(dI)
    bD = np.array(WATER_TYPES[wt]["bD"])
    with np.errstate(divide="ignore", invalid="ignore"):
        z = np.log(dI/min_contrast)/bD
    return np.maximum(z, 0)

rows = []
for wt in WATER_TYPES:
    z = detection_range(wt)
    rows.append(dict(water_type=wt, range_R_m=round(z[0], 2), range_G_m=round(z[1], 2),
                     range_B_m=round(z[2], 2), usable_range_m=round(z.max(), 2),
                     best_channel="RGB"[int(np.argmax(z))]))
RANGE_BUDGET = pd.DataFrame(rows).set_index("water_type")
display(RANGE_BUDGET.style.set_caption(
    f"Optical detection-range budget at a {MIN_CONTRAST:.3f} contrast floor (3x sensor noise)"))

fig, ax = plt.subplots(1, 3, figsize=(13, 3.3))
zz = np.linspace(0.05, 10, 300)
for wt, c in zip(WATER_TYPES, ["#1F3A5F", "#2E7D82", "#C0392B", "#8E6C1F"]):
    con = np.array([np.abs(uw_forward(Ifg, z, wt)-uw_forward(Ibg, z, wt)).max() for z in zz])
    ax[0].semilogy(zz, con, color=c, label=wt)
ax[0].axhline(MIN_CONTRAST, color="k", ls="--", lw=1, label="3-sigma sensor floor")
ax[0].set_xlabel("range z (m)"); ax[0].set_ylabel("best-channel contrast")
ax[0].legend(fontsize=6); ax[0].set_title("Contrast decays exponentially with range")
for wt, c in zip(WATER_TYPES, ["#1F3A5F", "#2E7D82", "#C0392B", "#8E6C1F"]):
    im = np.array([uw_forward(Ifg, z, wt) for z in zz])
    ax[1].plot(zz, im[:, 0], color=c, lw=1.1)
    ax[1].plot(zz, im[:, 1], color=c, lw=1.1, ls="--")
ax[1].set_xlabel("range z (m)"); ax[1].set_ylabel("observed radiance")
ax[1].set_title("Solid = red channel, dashed = green\n(red dies first; the veil takes over)",
                fontsize=8)
strip = np.zeros((60, 300, 3))
for i, z in enumerate(zz):
    strip[:30, i] = np.clip(uw_forward(Ifg, z, "coastal 1 (typical)"), 0, 1)
    strip[30:, i] = np.clip(uw_forward(Ibg, z, "coastal 1 (typical)"), 0, 1)
ax[2].imshow(strip, aspect="auto", extent=[0, 10, 0, 1])
ax[2].set_xlabel("range z (m)"); ax[2].set_yticks([])
ax[2].set_title("Object (top) vs background (bottom), coastal 1", fontsize=8)
ax[2].grid(False)
plt.tight_layout(); plt.show()

print("Read the table as a siting spec. In estuarine BC water the usable optical range is")
print(f"about {RANGE_BUDGET.loc['estuarine (BC wet)','usable_range_m']:.1f} m. A camera mounted "
      "5 m from the area of interest will see nothing")
print("on a wet-season day, and no detector architecture changes that. Move the camera,")
print("add illumination, or accept a duty cycle tied to turbidity — those are the options.")


### 2.3 Turbidity is the operational variable, so convert it to range

Field crews measure **turbidity in NTU**, not attenuation coefficients. The screening link
$c \approx c_w + k\,\mathrm{NTU}$ (with $k$ site-calibrated, typically 0.1–0.2 m⁻¹ per NTU)
converts a sensor reading into a **live detection range**, which is what lets the edge device
know when to stop trusting itself. This is the concrete coupling between Workstream A and the
turbidity sensor in Workstream B.

In [ ]:
def ntu_to_attenuation(ntu, c_water=0.15, k=0.14):
    return c_water + k*np.asarray(ntu, float)

def ntu_to_range(ntu, dI=0.30, min_contrast=MIN_CONTRAST):
    return np.log(dI/min_contrast)/np.maximum(ntu_to_attenuation(ntu), 1e-6)

ntu = np.array([0.3, 1, 2, 5, 10, 20, 50, 100])
t = pd.DataFrame(dict(NTU=ntu, c_per_m=ntu_to_attenuation(ntu).round(3),
                      detection_range_m=ntu_to_range(ntu).round(2)))
t["camera_at_2m"] = np.where(t.detection_range_m >= 2.0, "usable", "BLIND")
t["camera_at_5m"] = np.where(t.detection_range_m >= 5.0, "usable", "BLIND")
display(t.set_index("NTU").style.set_caption("Turbidity -> attenuation -> usable optical range"))

fig, ax = plt.subplots(figsize=(6.6, 3.2))
nn = np.linspace(0.2, 100, 400)
ax.loglog(nn, ntu_to_range(nn), color="#1F3A5F", lw=1.6)
for d, c in [(2.0, "#2E7D82"), (5.0, "#C0392B")]:
    ax.axhline(d, ls="--", color=c, lw=1, label=f"camera standoff {d:.0f} m")
    crit = np.exp((np.log(0.30/MIN_CONTRAST)/d - 0.15)/0.14) if False else \
           (np.log(0.30/MIN_CONTRAST)/d - 0.15)/0.14
    ax.axvline(crit, ls=":", color=c, lw=1)
    ax.annotate(f"blind above {crit:.0f} NTU", (crit, d*1.15), fontsize=7, color=c)
ax.set_xlabel("turbidity (NTU)"); ax.set_ylabel("detection range (m)")
ax.legend(fontsize=7); ax.set_title("Live turbidity tells the edge device when to abstain")
plt.tight_layout(); plt.show()
print("This curve is a RUNTIME input, not a report figure. The edge device reads the turbidity")
print("sensor, computes the range, compares it to the camera standoff, and if the target zone")
print("is outside the usable range it emits 'unknown / optically blind' instead of a negative.")
print("That is the difference between a monitoring system and a camera that lies quietly.")

## Part 3 — A scene generator with exact ground truth

Public datasets give you images and boxes. They do **not** tell you the range to each animal,
the turbidity at that moment, or how occluded the target was — so you cannot say *why* the
detector failed, only that it did.

The generator below renders each frame through the Part 2 optics, so every object carries
**exact metadata**: range $z$, water type, illumination, occlusion fraction, and apparent
contrast. That turns "accuracy 0.83" into "recall is 0.95 inside 1.5 m and 0.31 beyond
3 m in coastal-1 water" — which is an engineering specification.

Swap in the real loader with `RUN_REAL_DATA=True`; everything downstream is agnostic to the
source because it consumes a common record schema.

In [ ]:
H_IMG, W_IMG = 96, 128
CLASSES = ["empty", "fish", "non_fish"]      # Level 1 of the staged hierarchy
CLASS_STRUCTURE = {
  "level_1": ["empty", "fish", "non_fish"],
  "level_2": ["salmonid_like", "flatfish_like", "crustacean", "jelly", "unknown_organism"],
  "level_3": ["normal", "surface_congregation", "erratic_swimming", "mortality_indicator",
              "vessel_or_human", "debris_or_obstruction"],
}
LABEL_MAP = {"fish": "fish", "Fish": "fish", "small_fish": "fish", "fish_school": "fish",
             "crab": "non_fish", "jellyfish": "non_fish", "starfish": "non_fish",
             "shrimp": "non_fish", "background": "empty", "empty": "empty", "": "empty"}
def map_label(native): return LABEL_MAP.get(native, "unknown")

def _substrate(h, w, r):
    yy, xx = np.mgrid[0:h, 0:w]/max(h, w)
    base = np.stack([0.26+0.06*xx, 0.30+0.05*yy, 0.28+0.04*(1-yy)], -1)
    for _ in range(9):                                  # cobble / kelp texture
        cy, cx = r.integers(0, h), r.integers(0, w)
        rad = r.uniform(4, 16)
        m = (xx*max(h, w)-cx)**2 + (yy*max(h, w)-cy)**2 < rad**2
        base[m] += r.normal(0, 0.05, 3)
    return np.clip(base + r.normal(0, 0.012, (h, w, 3)), 0, 1)

def _draw_animal(canvas, zbuf, cx, cy, L_px, ang, elong, radiance, z, r, tail=True):
    h, w = canvas.shape[:2]
    a = L_px/2; b = a/elong
    yy, xx = np.mgrid[0:h, 0:w]
    X = (xx-cx)*np.cos(ang) + (yy-cy)*np.sin(ang)
    Y = -(xx-cx)*np.sin(ang) + (yy-cy)*np.cos(ang)
    body = (X/a)**2 + (Y/b)**2 <= 1
    if tail:
        tx = -a*0.95
        body |= (np.abs(X-tx) < a*0.30) & (np.abs(Y) < b*(1.6*np.abs(X-tx)/(a*0.30)+0.15))
    nearer = body & (z < zbuf)
    canvas[nearer] = radiance
    zbuf[nearer] = z
    return body, nearer

def render_frame(cls, water_type="coastal 1 (typical)", light=1.0, n_obj=None,
                 z_range=(0.6, 4.5), seed=None, motion_blur=0):
    '''Returns (image, records). Each record has exact range, contrast and occlusion.'''
    r = np.random.default_rng(seed) if seed is not None else RNG
    canvas = _substrate(H_IMG, W_IMG, r)*light
    zbuf = np.full((H_IMG, W_IMG), 1e9)
    recs = []
    n = (0 if cls == "empty" else (n_obj if n_obj is not None else r.integers(1, 4)))
    placed = []
    for _ in range(n):
        z = float(r.uniform(*z_range))
        L_px = float(np.clip(240/z*r.uniform(0.09, 0.17), 7, 60))   # apparent size ~ 1/z
        cx, cy = r.uniform(.1*W_IMG, .9*W_IMG), r.uniform(.15*H_IMG, .85*H_IMG)
        ang = r.uniform(0, np.pi)
        if cls == "fish":
            elong = r.uniform(3.0, 4.6)
            I_obj = np.array([r.uniform(.60, .82), r.uniform(.52, .70), r.uniform(.26, .42)])
        else:
            elong = r.uniform(1.0, 1.7)
            I_obj = np.array([r.uniform(.34, .52), r.uniform(.36, .56), r.uniform(.38, .60)])
        rad = uw_forward(I_obj*light, z, water_type)
        body, vis = _draw_animal(canvas, zbuf, cx, cy, L_px, ang, elong, rad, z, r)
        occ = 1 - vis.sum()/max(body.sum(), 1)
        if vis.sum() < 12:
            continue
        ys, xs = np.nonzero(vis)
        bg = uw_forward(np.array([.27, .31, .29])*light, z, water_type)
        recs.append(dict(cls=cls, z_m=z, L_px=L_px, elong=elong, occlusion=float(occ),
                         contrast=float(np.abs(rad-bg).max()),
                         bbox=(int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())),
                         area_px=int(vis.sum())))
        placed.append((cx, cy))
    # water column veil already applied per object; apply it to the substrate too
    far = zbuf > 1e8
    canvas[far] = uw_forward(canvas[far]/max(light, 1e-6), float(np.mean(z_range))*1.6, water_type)
    if motion_blur > 1:
        k = np.ones(motion_blur)/motion_blur
        canvas = np.stack([ndi.convolve1d(canvas[..., c], k, axis=1, mode="nearest")
                           for c in range(3)], -1)
    canvas = np.clip(canvas + r.normal(0, SENSOR_NOISE, canvas.shape), 0, 1)
    return canvas, recs

def build_dataset(n_clips=36, frames_per_clip=6, seed=0):
    '''Frames are generated in CLIPS; each clip fixes water type, light and camera pose,
    so the grouped split in Part 1 is meaningful rather than decorative.'''
    r = np.random.default_rng(seed)
    wts = list(WATER_TYPES)
    X, Y, META = [], [], []
    for c in range(n_clips):
        wt = wts[r.integers(0, len(wts))]
        light = float(r.uniform(0.45, 1.0))
        blur = int(r.choice([0, 0, 0, 3, 5]))
        for f in range(frames_per_clip):
            cls = str(r.choice(CLASSES, p=[.30, .45, .25]))
            img, recs = render_frame(cls, wt, light, seed=int(r.integers(1e9)),
                                     motion_blur=blur)
            if cls != "empty" and not recs:
                cls = "empty"
            X.append(img.astype(np.float32)); Y.append(CLASSES.index(cls))
            META.append(dict(clip=c, water_type=wt, light=light, blur=blur, cls=cls,
                             n_obj=len(recs),
                             mean_z=float(np.mean([q["z_m"] for q in recs])) if recs else np.nan,
                             min_z=float(np.min([q["z_m"] for q in recs])) if recs else np.nan,
                             max_contrast=float(np.max([q["contrast"] for q in recs])) if recs else 0.0,
                             max_occlusion=float(np.max([q["occlusion"] for q in recs])) if recs else 0.0,
                             recs=recs))
    return np.array(X), np.array(Y), pd.DataFrame(META)

t0 = time.time()
NC = 36 if RUN["FAST"] else 90
X, Y, META = build_dataset(n_clips=NC, frames_per_clip=6, seed=RUN["SEED"])
print(f"{len(X)} frames from {META['clip'].nunique()} clips in {time.time()-t0:.1f}s")
display(META.groupby(["water_type", "cls"]).size().unstack(fill_value=0))

fig, ax = plt.subplots(2, 4, figsize=(13, 5))
for a, wt in zip(ax[0], WATER_TYPES):
    im, rc = render_frame("fish", wt, 1.0, n_obj=3, seed=7)
    a.imshow(im); a.set_title(f"{wt}\nmax contrast {max([q['contrast'] for q in rc]):.3f}",
                              fontsize=7)
    for q in rc:
        x0, y0, x1, y1 = q["bbox"]
        a.add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, color="yellow", lw=.8))
    a.axis("off")
for a, (lab, kw) in zip(ax[1], [("bright, close", dict(light=1.0, z_range=(0.6, 1.4))),
                                ("dim", dict(light=0.45, z_range=(0.6, 2.0))),
                                ("distant", dict(light=0.9, z_range=(3.0, 4.5))),
                                ("motion blur", dict(light=0.9, motion_blur=7))]):
    im, rc = render_frame("fish", "coastal 1 (typical)", n_obj=3, seed=11, **kw)
    a.imshow(im); a.set_title(lab, fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()
print("Every box above carries an exact range, contrast and occlusion value. Part 5 uses them")
print("to report performance as a FUNCTION of condition rather than as a single number.")


## Part 4 — Detector: features chosen because of the physics, not despite it

Part 2 proved that backscatter destroys absolute brightness but preserves contrast. So the
feature set is built around **local contrast and shape**, all normalised by a robust
background estimate, plus an explicit **image-quality** channel that the abstention logic in
Part 6 consumes:

| Feature | Why |
|---|---|
| foreground area at a contrast-adaptive threshold | scale-free presence |
| elongation (PCA eigenvalue ratio) | fish vs crustacean/jelly is a shape question |
| normalised local contrast | survives the veil; absolute brightness does not |
| edge energy ratio (foreground / whole frame) | separates real objects from a uniform haze |
| red-channel deficit | red attenuates fastest, so it encodes range |
| **quality**: global contrast, estimated veil strength | tells the system when it is blind |

The CNN path (`RUN_REAL_DATA=True`) is the real upgrade; everything downstream — evaluation,
calibration, conformal sets, tracking, occupancy — is model-agnostic and works unchanged.

In [ ]:
def robust_bg(img):
    b = np.concatenate([img[:6, :].reshape(-1, 3), img[-6:, :].reshape(-1, 3),
                        img[:, :6].reshape(-1, 3), img[:, -6:].reshape(-1, 3)])
    return np.median(b, 0)

def quality(img):
    '''Image-quality channel: global contrast and an estimated veil fraction.'''
    g = img.mean(-1)
    p1, p99 = np.percentile(g, [1, 99])
    gc = float(p99-p1)
    veil = float(np.clip(1 - gc/0.55, 0, 1))          # low dynamic range => strong veil
    sharp = float(np.abs(np.diff(g, axis=1)).mean() + np.abs(np.diff(g, axis=0)).mean())
    return dict(global_contrast=gc, veil=veil, sharpness=sharp)

def features(img):
    bg = robust_bg(img)
    d = np.sqrt(((img-bg)**2).sum(-1))
    q = quality(img)
    thr = max(2.5*SENSOR_NOISE, 0.35*np.percentile(d, 99))   # contrast-adaptive, not absolute
    m = d > thr
    if m.sum() >= 12:
        m = ndi.binary_opening(m, np.ones((2, 2)))
    area = m.mean()
    if m.sum() >= 12:
        lab, n = ndi.label(m)
        sizes = ndi.sum(m, lab, range(1, n+1))
        big = (lab == (1+int(np.argmax(sizes))))
        ys, xs = np.nonzero(big)
        cov = np.cov(np.stack([xs-xs.mean(), ys-ys.mean()]))
        ev = np.clip(np.linalg.eigvalsh(cov), 1e-6, None)
        elong = float(np.sqrt(ev[1]/ev[0]))
        fg_rgb = img[big].mean(0)
        contrast = float(np.abs(fg_rgb-bg).max()/max(q["global_contrast"], 1e-3))
        red_def = float((fg_rgb[1]-fg_rgb[0])/max(fg_rgb.sum(), 1e-3))
        fill = float(big.sum()/max(m.sum(), 1))
        blob_frac = float(big.sum()/big.size)
    else:
        elong = contrast = red_def = fill = blob_frac = 0.0
    g = img.mean(-1)
    e_all = float(np.abs(np.diff(g, axis=1)).mean()+np.abs(np.diff(g, axis=0)).mean())
    e_fg = float(np.abs(np.diff(g*m, axis=1)).mean()+np.abs(np.diff(g*m, axis=0)).mean())
    edge_ratio = e_fg/max(e_all, 1e-6)
    return np.array([area, blob_frac, elong, contrast, red_def, fill, edge_ratio,
                     q["global_contrast"], q["veil"], q["sharpness"]])

FEATS = ["area", "blob_frac", "elong", "contrast", "red_deficit", "fill", "edge_ratio",
         "global_contrast", "veil", "sharpness"]

def softmax_fit(Xs, y, C=3, lr=.35, epochs=1500, l2=3e-4):
    n, d = Xs.shape; W = np.zeros((d, C)); b = np.zeros(C); Yh = np.eye(C)[y]
    for _ in range(epochs):
        z = Xs@W + b; z -= z.max(1, keepdims=True); e = np.exp(z); P = e/e.sum(1, keepdims=True)
        W -= lr*(Xs.T@(P-Yh)/n + l2*W); b -= lr*(P-Yh).mean(0)
    return W, b
def softmax_logits(Xs, W, b): return Xs@W + b
def softmax_prob(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e/e.sum(1, keepdims=True)

# ---------- grouped split by clip: train / calibration / test ----------------
clips = META["clip"].values
uc = np.unique(clips); RNG.shuffle(uc)
n_tr, n_cal = int(.55*len(uc)), int(.20*len(uc))
tr_c, cal_c, te_c = uc[:n_tr], uc[n_tr:n_tr+n_cal], uc[n_tr+n_cal:]
i_tr, i_cal, i_te = (np.isin(clips, tr_c), np.isin(clips, cal_c), np.isin(clips, te_c))
print(f"clips  train {len(tr_c)} | calibration {len(cal_c)} | test {len(te_c)}   (grouped, no overlap)")
print(f"frames train {i_tr.sum()} | calibration {i_cal.sum()} | test {i_te.sum()}")
print("The CALIBRATION split is separate from test because conformal prediction and")
print("temperature scaling both need held-out data that the test estimate has never seen.")

t0 = time.time()
F = np.array([features(im) for im in X])
print(f"feature extraction: {1000*(time.time()-t0)/len(X):.2f} ms/frame on this CPU")
mu, sd = F[i_tr].mean(0), F[i_tr].std(0)+1e-6
Z = (F-mu)/sd
Wm, bm = softmax_fit(Z[i_tr], Y[i_tr])
logits_te = softmax_logits(Z[i_te], Wm, bm); P_te = softmax_prob(logits_te)
print(f"\nbaseline test accuracy (grouped split): {(P_te.argmax(1)==Y[i_te]).mean():.3f}")

imp = pd.DataFrame(np.abs(Wm), index=FEATS, columns=CLASSES)
imp["total"] = imp.sum(1)
display(imp.sort_values("total", ascending=False).round(3)
        .style.set_caption("|weight| per feature — the contrast/shape features carry the signal"))

### 4.1 The CNN reference (Colab GPU, guarded)

Everything after this point consumes `(probabilities, labels, metadata)`. Swap the model and
nothing else changes — that is the whole point of keeping the evaluation harness separate
from the model.

In [ ]:
def train_cnn_reference(data_dir, epochs=8, img=128, batch=32, arch="mobilenet_v3_small"):
    '''Fine-tune a compact backbone. Run in Colab with a GPU and RUN_REAL_DATA=True.
    IMPORTANT: build the split with GroupShuffleSplit on clip id, not random_split.'''
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, Subset
    from torchvision import datasets, transforms, models
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tf_tr = transforms.Compose([
        transforms.Resize((img, img)),
        transforms.ColorJitter(0.35, 0.35, 0.35, 0.06),   # stands in for turbidity/light shift
        transforms.RandomHorizontalFlip(), transforms.ToTensor()])
    tf_te = transforms.Compose([transforms.Resize((img, img)), transforms.ToTensor()])
    ds = datasets.ImageFolder(data_dir, transform=tf_tr)
    groups = [os.path.basename(p).split("_")[0] for p, _ in ds.samples]   # clip id prefix
    from sklearn.model_selection import GroupShuffleSplit
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
    tr_i, te_i = next(gss.split(np.zeros(len(ds)), [l for _, l in ds.samples], groups))
    dl_tr = DataLoader(Subset(ds, tr_i), batch, shuffle=True, num_workers=2)
    ds_te = datasets.ImageFolder(data_dir, transform=tf_te)
    dl_te = DataLoader(Subset(ds_te, te_i), batch, num_workers=2)
    net = getattr(models, arch)(weights="DEFAULT")
    net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, len(ds.classes))
    net = net.to(dev)
    opt = torch.optim.AdamW(net.parameters(), 1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    # class weights: the event classes are rare, and unweighted CE will ignore them
    import collections
    cnt = collections.Counter([ds.samples[i][1] for i in tr_i])
    wts = torch.tensor([len(tr_i)/(len(cnt)*cnt[c]) for c in range(len(ds.classes))],
                       dtype=torch.float32, device=dev)
    lossf = nn.CrossEntropyLoss(weight=wts, label_smoothing=0.05)
    for ep in range(epochs):
        net.train()
        for xb, yb in dl_tr:
            xb, yb = xb.to(dev), yb.to(dev)
            opt.zero_grad(); lossf(net(xb), yb).backward(); opt.step()
        sch.step()
    net.eval(); Pl, Yl = [], []
    with torch.no_grad():
        for xb, yb in dl_te:
            Pl.append(torch.softmax(net(xb.to(dev)), 1).cpu().numpy()); Yl.append(yb.numpy())
    P, Yt = np.concatenate(Pl), np.concatenate(Yl)
    print(f"CNN ({arch}) grouped-split accuracy: {(P.argmax(1)==Yt).mean():.3f} on {dev}")
    return net, P, Yt

try:
    import torch; HAVE_TORCH = True
except Exception:
    HAVE_TORCH = False
print(f"PyTorch available: {HAVE_TORCH} | RUN_REAL_DATA: {RUN['RUN_REAL_DATA']}")
if HAVE_TORCH and RUN["RUN_REAL_DATA"]:
    print("Call train_cnn_reference('<imagefolder>') — note the GroupShuffleSplit and the")
    print("class weights; both matter more than the architecture choice.")
else:
    print("CNN path skipped. The baseline below carries the rest of the notebook; every")
    print("downstream component is model-agnostic and will accept CNN probabilities unchanged.")

## Part 5 — Evaluation as a measurement, with uncertainty and conditions

Three upgrades over a single accuracy number:

1. **Bootstrap confidence intervals, resampled by clip.** Resampling frames underestimates
   the interval, because frames within a clip are correlated. The cluster bootstrap is the
   honest version.
2. **Stratification by condition.** Recall as a function of range, turbidity and light is the
   engineering answer; the marginal number is a summary of a deployment that never happens.
3. **Precision–recall and AP**, because at low event prevalence the ROC curve is
   misleadingly flattering — a point Part 7 makes numerically.

In [ ]:
def class_metrics(y, p, labels=CLASSES):
    rows = []
    for i, L in enumerate(labels):
        tp = int(((p == i) & (y == i)).sum()); fp = int(((p == i) & (y != i)).sum())
        fn = int(((p != i) & (y == i)).sum())
        pr = tp/(tp+fp) if tp+fp else 0.0; rc = tp/(tp+fn) if tp+fn else 0.0
        rows.append(dict(cls=L, precision=pr, recall=rc,
                         f1=2*pr*rc/(pr+rc) if pr+rc else 0.0, support=int((y == i).sum())))
    return pd.DataFrame(rows).set_index("cls")

def confusion(y, p, labels=CLASSES):
    M = np.zeros((len(labels), len(labels)), int)
    for a, b in zip(y, p): M[a, b] += 1
    return pd.DataFrame(M, index=[f"true_{l}" for l in labels],
                        columns=[f"pred_{l}" for l in labels])

def cluster_bootstrap(y, p, groups, stat, n_boot=400, seed=0):
    '''Resample CLIPS with replacement, not frames.'''
    r = np.random.default_rng(seed); uc = np.unique(groups); out = []
    for _ in range(n_boot):
        pick = r.choice(uc, len(uc), replace=True)
        idx = np.concatenate([np.nonzero(groups == c)[0] for c in pick])
        try: out.append(stat(y[idx], p[idx]))
        except Exception: pass
    return np.percentile(out, [2.5, 50, 97.5])

def ap_101(scores, labels, n_gt):
    '''COCO-style 101-point interpolated average precision.'''
    o = np.argsort(-scores); lab = np.asarray(labels)[o]
    tp = np.cumsum(lab); fp = np.cumsum(1-lab)
    rec = tp/max(n_gt, 1); prec = tp/np.maximum(tp+fp, 1e-12)
    mprec = np.maximum.accumulate(prec[::-1])[::-1]
    out = [mprec[rec >= r].max() if (rec >= r).any() else 0.0 for r in np.linspace(0, 1, 101)]
    return float(np.mean(out))

# ---- VET the AP implementation on cases with known answers ------------------
assert abs(ap_101(np.array([.9, .8, .7]), np.array([1, 1, 1]), 3) - 1.0) < 1e-9
assert abs(ap_101(np.array([.9, .8, .7]), np.array([1, 1, 1]), 6) - 0.505) < 0.02
_ap_fp = ap_101(np.array([.95, .9, .8, .7]), np.array([0, 1, 1, 1]), 3)
assert 0.6 < _ap_fp < 0.9
print(f"[VET] AP: perfect=1.000 | half the GT missed={ap_101(np.array([.9,.8,.7]),np.array([1,1,1]),6):.3f}"
      f" | one FP first={_ap_fp:.3f}  -> PASS")

pred_te = P_te.argmax(1); y_te = Y[i_te]; g_te = clips[i_te]
print("\nPer-class metrics with 95% cluster-bootstrap CIs (resampled by clip):")
rows = []
for i, L in enumerate(CLASSES):
    rec = lambda yy, pp, i=i: ((pp == i) & (yy == i)).sum()/max((yy == i).sum(), 1)
    prc = lambda yy, pp, i=i: ((pp == i) & (yy == i)).sum()/max((pp == i).sum(), 1)
    lo_r, md_r, hi_r = cluster_bootstrap(y_te, pred_te, g_te, rec,
                                         n_boot=300 if RUN["FAST"] else 1000)
    lo_p, md_p, hi_p = cluster_bootstrap(y_te, pred_te, g_te, prc,
                                         n_boot=300 if RUN["FAST"] else 1000)
    rows.append(dict(cls=L, recall=md_r, recall_CI=f"[{lo_r:.2f}, {hi_r:.2f}]",
                     precision=md_p, precision_CI=f"[{lo_p:.2f}, {hi_p:.2f}]",
                     support=int((y_te == i).sum())))
display(pd.DataFrame(rows).set_index("cls").round(3))
print("Compare the CI widths with a naive frame-level bootstrap: the clip bootstrap is")
print("typically 1.5-3x wider, and it is the one that survives peer review.")

# ---- performance as a function of condition --------------------------------
Mte = META[i_te].copy(); Mte["correct"] = (pred_te == y_te)
Mte["pred"] = [CLASSES[k] for k in pred_te]
pos = Mte[Mte.cls != "empty"].copy()
pos["z_bin"] = pd.cut(pos.min_z, [0, 1.0, 1.75, 2.75, 5.0])
pos["c_bin"] = pd.cut(pos.max_contrast, [0, .02, .05, .10, 1.0])

fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.3))
by_wt = Mte.groupby("water_type").correct.agg(["mean", "size"])
ax[0].barh(by_wt.index, by_wt["mean"], color="#1F3A5F")
ax[0].set_xlabel("accuracy"); ax[0].set_xlim(0, 1)
ax[0].set_title("by water type"); ax[0].tick_params(labelsize=6.5)
if pos.z_bin.notna().any():
    bz = pos.groupby("z_bin", observed=True).correct.mean()
    ax[1].bar([str(s) for s in bz.index], bz.values, color="#2E7D82")
    ax[1].set_ylabel("recall on frames with an animal"); ax[1].set_ylim(0, 1)
    ax[1].set_title("by range to nearest animal (m)"); ax[1].tick_params(labelsize=6.5, rotation=20)
if pos.c_bin.notna().any():
    bc = pos.groupby("c_bin", observed=True).correct.mean()
    ax[2].bar([str(s) for s in bc.index], bc.values, color="#C0392B")
    ax[2].axvline(-.5, color="k", lw=0)
    ax[2].set_ylim(0, 1); ax[2].set_title("by apparent contrast")
    ax[2].tick_params(labelsize=6.5, rotation=20)
plt.tight_layout(); plt.show()

print("This is the deliverable the graduate student needs for Phase 3: not 'the model is 0.83'")
print("but 'recall exceeds 0.9 while apparent contrast is above ~0.05, and collapses below it'.")
print("Combined with the Part 2 range budget, that becomes a camera placement and duty cycle.")
display(confusion(y_te, pred_te))

## Part 6 — Calibration and abstention with a guarantee

### 6.1 A softmax output is not a probability

Confidence is only meaningful if 0.9 means "right 90 % of the time". **Expected calibration
error** measures the gap:

$$\mathrm{ECE} = \sum_{m=1}^{M}\frac{|B_m|}{n}\Big|\,\mathrm{acc}(B_m) - \mathrm{conf}(B_m)\Big|$$

**Temperature scaling** fixes most of it with one parameter fitted on held-out data:
$\hat p = \mathrm{softmax}(z/T)$. It cannot change any prediction (softmax is monotone in
$z/T$ for a fixed argmax), so **accuracy is provably unchanged** — a property we assert below.

### 6.2 Abstention that comes with a coverage guarantee

A confidence threshold gives you a trade-off curve with no promise attached. **Split
conformal prediction** gives a promise. Using the non-conformity score $s = 1-\hat p_y$ on a
held-out calibration set of size $n$, take

$$\hat q = s_{(k)},\qquad k=\lceil (n+1)(1-\alpha)\rceil$$

and emit the set $\{c : 1-\hat p_c \le \hat q\}$. Under exchangeability this satisfies

$$1-\alpha \;\le\; \mathbb{P}\big(y_{test} \in \mathcal{C}(x_{test})\big) \;<\; 1-\alpha+\tfrac{1}{n+1}$$

**distribution-free and finite-sample** — no assumption about the model being good. Sets of
size 0 mean "nothing is plausible" (out of distribution → escalate); size ≥ 2 means
"genuinely ambiguous" (→ human review). That maps exactly onto the charter's requirement that
low confidence must not silently produce a safe decision.

The guarantee is **marginal**, not conditional: it holds on average over the distribution the
calibration set came from. Under the distribution shift in Part 12 it degrades, and we measure
by how much rather than assuming it away.

In [ ]:
def ece(P, y, M=15):
    conf = P.max(1); corr = (P.argmax(1) == y).astype(float)
    bins = np.linspace(0, 1, M+1); e = 0.0
    for i in range(M):
        m = (conf > bins[i]) & (conf <= bins[i+1])
        if m.sum(): e += m.mean()*abs(corr[m].mean()-conf[m].mean())
    return float(e)

def fit_temperature(logits, y, grid=np.linspace(0.3, 6.0, 300)):
    best, bt = np.inf, 1.0
    for T in grid:
        z = logits/T; z = z - z.max(1, keepdims=True)
        nll = -(z[np.arange(len(y)), y] - np.log(np.exp(z).sum(1))).mean()
        if nll < best: best, bt = nll, T
    return float(bt)

logits_cal = softmax_logits(Z[i_cal], Wm, bm); y_cal = Y[i_cal]
T_star = fit_temperature(logits_cal, y_cal)
P_te_T = softmax_prob(logits_te/T_star)

e0, e1 = ece(P_te, y_te), ece(P_te_T, y_te)
assert (P_te.argmax(1) == P_te_T.argmax(1)).all(), "temperature scaling changed a prediction"
print(f"fitted temperature T = {T_star:.3f}  ({'overconfident' if T_star>1 else 'underconfident'} before scaling)")
print(f"ECE  before {e0:.4f}  ->  after {e1:.4f}   ({100*(1-e1/max(e0,1e-9)):.0f}% reduction)")
print(f"[VET] accuracy identical before/after: {(P_te.argmax(1)==y_te).mean():.4f} -> "
      f"{(P_te_T.argmax(1)==y_te).mean():.4f}  -> PASS")

def reliability(P, y, M=10):
    conf = P.max(1); corr = (P.argmax(1) == y).astype(float)
    bins = np.linspace(0, 1, M+1); xs, ys, ws = [], [], []
    for i in range(M):
        m = (conf > bins[i]) & (conf <= bins[i+1])
        if m.sum() >= 3:
            xs.append(conf[m].mean()); ys.append(corr[m].mean()); ws.append(m.sum())
    return np.array(xs), np.array(ys), np.array(ws)

def conformal_qhat(s_cal, alpha):
    n = len(s_cal); k = min(int(np.ceil((n+1)*(1-alpha))), n)
    return float(np.sort(s_cal)[k-1])

s_cal = 1 - softmax_prob(logits_cal/T_star)[np.arange(len(y_cal)), y_cal]
CONF_RESULTS = []
for alpha in [0.01, 0.05, 0.10, 0.20]:
    q = conformal_qhat(s_cal, alpha)
    sets = (1 - P_te_T) <= q
    cov = float(sets[np.arange(len(y_te)), y_te].mean())
    sz = sets.sum(1)
    CONF_RESULTS.append(dict(alpha=alpha, target=1-alpha, empirical_coverage=cov,
                             mean_set_size=float(sz.mean()),
                             singleton_pct=100*float((sz == 1).mean()),
                             ambiguous_pct=100*float((sz >= 2).mean()),
                             ood_empty_pct=100*float((sz == 0).mean()), q_hat=q))
CONF = pd.DataFrame(CONF_RESULTS)
display(CONF.round(4).style.set_caption(
    "Split conformal prediction: empirical coverage should land on the target"))

fig, ax = plt.subplots(1, 3, figsize=(13, 3.3))
for P_, lab, c in [(P_te, "raw", "#C0392B"), (P_te_T, f"T={T_star:.2f}", "#1F3A5F")]:
    xs, ys, ws = reliability(P_, y_te)
    ax[0].plot(xs, ys, "o-", color=c, label=f"{lab} (ECE {ece(P_, y_te):.3f})", ms=4)
ax[0].plot([0, 1], [0, 1], "k--", lw=.8)
ax[0].set_xlabel("mean confidence"); ax[0].set_ylabel("empirical accuracy")
ax[0].legend(fontsize=7); ax[0].set_title("Reliability diagram")
ax[1].plot(CONF.target, CONF.empirical_coverage, "o-", color="#2E7D82", ms=5)
ax[1].plot([.78, 1], [.78, 1], "k--", lw=.8)
ax[1].set_xlabel("target coverage 1-alpha"); ax[1].set_ylabel("empirical coverage")
ax[1].set_title("Conformal guarantee holds on held-out data")
thr = np.linspace(0.34, 0.999, 60)
cov = [(P_te_T.max(1) >= t).mean() for t in thr]
accк = [((P_te_T.argmax(1) == y_te)[P_te_T.max(1) >= t]).mean()
        if (P_te_T.max(1) >= t).any() else np.nan for t in thr]
ax[2].plot(cov, accк, color="#8E6C1F", lw=1.5)
ax[2].set_xlabel("coverage (fraction not abstained)"); ax[2].set_ylabel("accuracy on kept")
ax[2].set_title("Risk-coverage curve")
plt.tight_layout(); plt.show()

print("Three distinct outputs, and the system must treat them differently:")
print("  singleton set -> a confident call, still only evidence")
print("  set size >= 2 -> genuinely ambiguous -> route to human review")
print("  EMPTY set     -> nothing is plausible -> out-of-distribution -> escalate, never 'empty frame'")
print("\nThe empty set is the one that matters. A plain argmax classifier CANNOT produce it;")
print("it will confidently name a class for an image of a novel organism, a fouled lens, or a")
print("diver's torch. Conformal prediction makes 'I have never seen this' a first-class output.")

## Part 7 — Choosing the operating point: base rates and asymmetric costs

### 7.1 The arithmetic that kills monitoring projects

At an event prevalence $\pi$, the precision of a detector with recall $r$ and false-positive
rate $f$ is

$$\mathrm{precision} = \frac{\pi r}{\pi r + (1-\pi) f}$$

When $\pi \sim 10^{-4}$ and $f = 10^{-2}$, precision is under 1 % — **99 of every 100 alerts
are wrong**, even with 90 % recall. Operators stop reading alerts within a week, and the
system is worse than nothing because it also consumes the attention that would have noticed
a real event.

Recall is never the binding constraint for rare events. **The false-positive rate multiplied
by the enormous negative class is.**

### 7.2 The operating point is a decision, so write down the costs

$$\mathbb{E}[\text{cost}](\tau) \;=\; C_{FN}\,\pi\,(1-r(\tau)) \;+\; C_{FP}\,(1-\pi)\,f(\tau)$$

The charter says to prioritise recall for safety-relevant events even at the price of false
positives, which is a statement that $C_{FN} \gg C_{FP}$. Making that ratio **explicit and
auditable** is better practice than hiding it inside a threshold someone picked by eye.

In [ ]:
FRAMES_PER_DAY = 24*3600/2        # one analysed frame every 2 s

def precision_at(prev, recall, fpr):
    tp = prev*recall; fp = (1-prev)*fpr
    return tp/max(tp+fp, 1e-15)

rows = []
for prev in [1e-2, 1e-3, 1e-4]:
    for rec, fpr in [(0.95, 0.05), (0.90, 0.01), (0.80, 0.001), (0.60, 0.0001)]:
        p = precision_at(prev, rec, fpr)
        alerts = (prev*rec + (1-prev)*fpr)*FRAMES_PER_DAY
        rows.append(dict(prevalence=prev, recall=rec, FPR=fpr, precision=round(p, 4),
                         alerts_per_day=round(alerts), true_alerts_per_day=round(prev*rec*FRAMES_PER_DAY, 2),
                         wrong_per_true=round((1-p)/max(p, 1e-9), 1)))
BASE = pd.DataFrame(rows)
display(BASE.pivot(index=["recall", "FPR"], columns="prevalence", values="precision")
        .style.background_gradient(cmap="RdYlGn", vmin=0, vmax=0.5)
        .format("{:.4f}").set_caption("Precision by prevalence — the base rate dominates"))
print(BASE[BASE.prevalence == 1e-4].to_string(index=False))
print("\nAt 1-in-10,000 prevalence, a 90%-recall / 1%-FPR detector produces about "
      f"{BASE[(BASE.prevalence==1e-4)&(BASE.FPR==0.01)].alerts_per_day.iloc[0]:.0f} alerts a day,")
print(f"of which {BASE[(BASE.prevalence==1e-4)&(BASE.FPR==0.01)].true_alerts_per_day.iloc[0]:.2f} are real.")

# ---- cost-optimal threshold on the real scores ------------------------------
is_event = (y_te != CLASSES.index("empty")).astype(int)
score = 1 - P_te_T[:, CLASSES.index("empty")]          # P(something present)
order = np.argsort(-score)
tp = np.cumsum(is_event[order]); fp = np.cumsum(1-is_event[order])
rec_c = tp/max(is_event.sum(), 1); fpr_c = fp/max((1-is_event).sum(), 1)
thr_c = score[order]

def expected_cost(prev, C_FN, C_FP):
    return C_FN*prev*(1-rec_c) + C_FP*(1-prev)*fpr_c

fig, ax = plt.subplots(1, 3, figsize=(13, 3.3))
ax[0].plot(fpr_c, rec_c, color="#1F3A5F", lw=1.5)
ax[0].plot([0, 1], [0, 1], "k--", lw=.7)
ax[0].set_xlabel("false-positive rate"); ax[0].set_ylabel("recall")
ax[0].set_title(f"ROC (AUC = {np.trapezoid(rec_c, fpr_c):.3f})")
prec_c = tp/np.maximum(tp+fp, 1e-9)
ax[1].plot(rec_c, prec_c, color="#2E7D82", lw=1.5)
ax[1].axhline(is_event.mean(), color="k", ls="--", lw=.7, label="prevalence (random baseline)")
ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].legend(fontsize=7)
ax[1].set_title(f"Precision-recall (AP = {ap_101(score, is_event, is_event.sum()):.3f})")
CHOSEN = {}
for ratio, c in [(1, "#8E6C1F"), (10, "#2E7D82"), (100, "#C0392B")]:
    ec = expected_cost(0.01, ratio, 1.0)
    k = int(np.argmin(ec))
    ax[2].plot(thr_c, ec/ec.max(), color=c, lw=1.3, label=f"C_FN/C_FP = {ratio}")
    ax[2].plot(thr_c[k], ec[k]/ec.max(), "o", color=c, ms=6)
    CHOSEN[ratio] = dict(threshold=float(thr_c[k]), recall=float(rec_c[k]),
                         fpr=float(fpr_c[k]), precision=float(prec_c[k]))
ax[2].set_xlabel("decision threshold"); ax[2].set_ylabel("expected cost (normalised)")
ax[2].legend(fontsize=7); ax[2].set_title("Cost-optimal operating point")
plt.tight_layout(); plt.show()
display(pd.DataFrame(CHOSEN).T.round(4).rename_axis("C_FN / C_FP")
        .style.set_caption("The threshold is a consequence of the cost ratio, not a free parameter"))
print("Note how the ROC curve looks excellent while the PR curve is modest. On rare events")
print("ROC/AUC flatters a detector, because the huge true-negative count swamps the FPR axis.")
print("Report AP and precision at the chosen operating point. Never report AUC alone.")

## Part 8 — Tracking: the cheapest large gain available

### 8.1 Why aggregate

A per-frame detector at false-positive rate $f$ that must fire on $k$ of $n$ consecutive
frames before raising an alert has an effective false-positive rate

$$f_{eff} \;=\; \sum_{j=k}^{n}\binom{n}{j} f^{j}(1-f)^{n-j}$$

For $f=0.01$, $n=5$, $k=3$ this is about $10^{-5}$ — a **thousand-fold** reduction — while
recall falls only modestly because a real animal persists across frames and a noise blob does
not. Temporal aggregation is a better investment than almost any model improvement.

The tracker also fixes a subtler problem: **counting**. Without identity, one fish swimming
past a camera for 40 frames is 40 detections. Abundance estimated from raw detections is
nonsense; it must be estimated from **tracks**.

### 8.2 The tracker

Constant-velocity Kalman filter per track, Hungarian assignment on Mahalanobis distance with
a $\chi^2_2$ gate at 99 %, track birth after confirmation and death after $N$ misses. Scored
with MOTA, IDF1 and HOTA, because they disagree and the disagreement is diagnostic:

$$\mathrm{MOTA}=1-\frac{FN+FP+IDSW}{|gtDet|},\qquad
\mathrm{IDF1}=\frac{IDTP}{IDTP+\tfrac12 IDFN+\tfrac12 IDFP},\qquad
\mathrm{HOTA}_\alpha=\sqrt{\mathrm{DetA}_\alpha\cdot\mathrm{AssA}_\alpha}$$

MOTA correlates strongly with detection quality and weakly with association; IDF1 is the
opposite; HOTA balances them. For counting, **association is what you care about**, so IDF1
and AssA are the numbers that matter — and a tracker chosen on MOTA alone can be the wrong one.

In [ ]:
class KalmanTrack:
    '''Constant-velocity model, state = [x, y, vx, vy].'''
    def __init__(self, z0, dt=1.0, q=1.0, r=4.0):
        self.x = np.array([z0[0], z0[1], 0., 0.])
        self.P = np.diag([r, r, 100., 100.])
        self.F = np.array([[1,0,dt,0],[0,1,0,dt],[0,0,1,0],[0,0,0,1]], float)
        G = np.array([[dt*dt/2,0],[0,dt*dt/2],[dt,0],[0,dt]])
        self.Q = G @ (q*np.eye(2)) @ G.T
        self.H = np.array([[1,0,0,0],[0,1,0,0]], float)
        self.R = r*np.eye(2); self.miss = 0; self.hits = 1
    def predict(self):
        self.x = self.F @ self.x; self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x[:2]
    def gate_dist(self, z):
        S = self.H @ self.P @ self.H.T + self.R
        y = np.asarray(z) - self.H @ self.x
        return float(y @ np.linalg.inv(S) @ y)
    def update(self, z):
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ (np.asarray(z) - self.H @ self.x)
        self.P = (np.eye(4) - K @ self.H) @ self.P
        self.miss = 0; self.hits += 1

CHI2_GATE = 9.21          # chi2 with 2 dof at 99%

def track_sequence(detections, max_miss=4, min_hits=2):
    tracks, nid, assoc = {}, 0, []
    for t, Zt in enumerate(detections):
        for tr in tracks.values(): tr.predict()
        ids = list(tracks); used = set()
        if ids and len(Zt):
            C = np.full((len(ids), len(Zt)), 1e6)
            for a, k in enumerate(ids):
                for b, z in enumerate(Zt):
                    d = tracks[k].gate_dist(z[:2])
                    if d < CHI2_GATE: C[a, b] = d
            r_, c_ = linear_sum_assignment(C)
            for a, b in zip(r_, c_):
                if C[a, b] < CHI2_GATE:
                    tracks[ids[a]].update(Zt[b][:2]); used.add(b)
                    assoc.append((t, ids[a], int(Zt[b][2])))
        for b in range(len(Zt)):
            if b not in used:
                tracks[nid] = KalmanTrack(Zt[b][:2]); assoc.append((t, nid, int(Zt[b][2])))
                nid += 1
        for k in list(tracks):
            if k not in [a[1] for a in assoc if a[0] == t]:
                tracks[k].miss += 1
                if tracks[k].miss > max_miss: del tracks[k]
    return np.array(assoc) if assoc else np.zeros((0, 3), int), nid

def simulate_detections(T=70, N=4, p_det=0.88, clutter=0.10, sigma=2.2, seed=0):
    r = np.random.default_rng(seed)
    p0 = r.uniform(20, 180, (N, 2)); v = r.uniform(-2.5, 2.5, (N, 2))
    dets, n_gt = [], np.full(T, N)
    for t in range(T):
        Z = []
        for i in range(N):
            if r.random() < p_det:
                Z.append([*(p0[i]+v[i]*t + r.normal(0, sigma, 2)), i])
        if r.random() < clutter:
            Z.append([*r.uniform(0, 200, 2), -1])
        dets.append(np.array(Z) if Z else np.zeros((0, 3)))
    return dets, n_gt

def mot_metrics(A, gt_total):
    if len(A) == 0: return dict(MOTA=0, IDF1=0, DetA=0, AssA=0, HOTA=0, TP=0, FP=0, FN=gt_total, IDSW=0)
    TP = int((A[:, 2] >= 0).sum()); FP = int((A[:, 2] < 0).sum()); FN = gt_total - TP
    idsw, last = 0, {}
    for t, tid, gid in A[np.lexsort((A[:, 0], A[:, 2]))]:
        if gid < 0: continue
        if gid in last and last[gid] != tid: idsw += 1
        last[gid] = tid
    MOTA = 1 - (FN+FP+idsw)/max(gt_total, 1)
    tids = np.unique(A[A[:, 2] >= 0, 1]); gids = np.unique(A[A[:, 2] >= 0, 2])
    M = np.array([[((A[:, 2] == g) & (A[:, 1] == tt)).sum() for tt in tids] for g in gids])
    r_, c_ = linear_sum_assignment(-M); IDTP = int(M[r_, c_].sum())
    IDFN, IDFP = gt_total-IDTP, len(A)-IDTP
    IDF1 = IDTP/(IDTP + .5*IDFN + .5*IDFP)
    DetA = TP/max(TP+FN+FP, 1)
    ass = []
    for t, tt, g in A:
        if g < 0: continue
        TPA = ((A[:, 1] == tt) & (A[:, 2] == g)).sum()
        FNA = ((A[:, 2] == g) & (A[:, 1] != tt)).sum()
        FPA = ((A[:, 1] == tt) & (A[:, 2] != g)).sum()
        ass.append(TPA/max(TPA+FNA+FPA, 1))
    AssA = float(np.mean(ass)) if ass else 0.0
    return dict(TP=TP, FP=FP, FN=FN, IDSW=idsw, MOTA=MOTA, IDF1=IDF1,
                DetA=DetA, AssA=AssA, HOTA=float(np.sqrt(DetA*AssA)))

# ---- VET: a perfect tracker must score 1.0 on every metric ------------------
T_, N_ = 70, 4
perfect = np.array([(t, i, i) for t in range(T_) for i in range(N_)])
mp = mot_metrics(perfect, T_*N_)
assert abs(mp["MOTA"]-1) < 1e-9 and abs(mp["HOTA"]-1) < 1e-9 and abs(mp["IDF1"]-1) < 1e-9
print(f"[VET] perfect tracker -> MOTA={mp['MOTA']:.3f} IDF1={mp['IDF1']:.3f} "
      f"HOTA={mp['HOTA']:.3f}  -> PASS")

rows = []
for p_det, clut, sig in [(0.95, 0.05, 1.5), (0.88, 0.10, 2.2), (0.70, 0.25, 4.0), (0.55, 0.40, 6.0)]:
    dets, n_gt = simulate_detections(p_det=p_det, clutter=clut, sigma=sig, seed=2)
    A, ntr = track_sequence(dets)
    m = mot_metrics(A, int(n_gt.sum()))
    m.update(dict(p_det=p_det, clutter=clut, sigma_px=sig, tracks_created=ntr,
                  true_targets=4, count_error_pct=100*(ntr-4)/4))
    rows.append(m)
MOT = pd.DataFrame(rows).set_index(["p_det", "clutter", "sigma_px"])
display(MOT[["TP", "FP", "FN", "IDSW", "MOTA", "IDF1", "DetA", "AssA", "HOTA",
             "tracks_created", "count_error_pct"]].round(3)
        .style.set_caption("Tracking quality as detection degrades"))
print("Watch `tracks_created` against `true_targets = 4`. As detection degrades the tracker")
print("fragments and over-counts. That fragmentation, not the detector's recall, is what")
print("corrupts an abundance estimate — and MOTA barely registers it while AssA does.")

In [ ]:
# ---- temporal aggregation: k-of-n voting, theory vs simulation --------------
from math import comb
def f_eff(f, n, k):
    return sum(comb(n, j)*f**j*(1-f)**(n-j) for j in range(k, n+1))
def r_eff(r, n, k):
    return sum(comb(n, j)*r**j*(1-r)**(n-j) for j in range(k, n+1))

rows = []
for n, k in [(1, 1), (3, 2), (5, 3), (7, 4), (9, 5)]:
    for f, r in [(0.01, 0.90), (0.05, 0.90)]:
        fe, re_ = f_eff(f, n, k), r_eff(r, n, k)
        rows.append(dict(window=n, k_of_n=f"{k}/{n}", per_frame_FPR=f, per_frame_recall=r,
                         eff_FPR=fe, eff_recall=re_, FPR_reduction=f"{f/max(fe,1e-12):,.0f}x",
                         precision_at_1e4=precision_at(1e-4, re_, fe),
                         alerts_per_day=(1e-4*re_+(1-1e-4)*fe)*FRAMES_PER_DAY))
AGG = pd.DataFrame(rows)
display(AGG[AGG.per_frame_FPR == 0.01].round(6).set_index("k_of_n")
        .style.set_caption("k-of-n temporal voting at a 1% per-frame false-positive rate"))

# simulation check of the binomial theory
r_ = np.random.default_rng(0); N = 200000
sim = []
for n, k in [(3, 2), (5, 3), (7, 4)]:
    draws = (r_.random((N, n)) < 0.01).sum(1) >= k
    sim.append(dict(k_of_n=f"{k}/{n}", theory=f_eff(0.01, n, k), simulated=draws.mean()))
s = pd.DataFrame(sim)
display(s.assign(rel_err=lambda d: (d.simulated-d.theory).abs()/d.theory).round(8))
assert (np.abs(s.simulated-s.theory) < 3*np.sqrt(s.theory*(1-s.theory)/N) + 1e-9).all()
print("[VET] simulated k-of-n false-positive rates match the binomial theory -> PASS")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
sub = AGG[AGG.per_frame_FPR == 0.01]
ax[0].semilogy(sub.window, sub.eff_FPR, "o-", color="#C0392B", label="effective FPR")
ax[0].semilogy(sub.window, 1-sub.eff_recall, "s-", color="#1F3A5F", label="effective miss rate")
ax[0].set_xlabel("aggregation window n (majority vote)"); ax[0].legend(fontsize=7)
ax[0].set_title("FPR falls thousands-fold; recall barely moves")
ax[1].plot(sub.window, sub.alerts_per_day, "o-", color="#2E7D82")
ax[1].set_yscale("log"); ax[1].set_xlabel("aggregation window n")
ax[1].set_ylabel("alerts/day at 1e-4 prevalence")
ax[1].axhline(20, color="k", ls="--", lw=.8)
ax[1].annotate("~20/day: what an operator will actually read", (3, 24), fontsize=7)
ax[1].set_title("Alert budget")
plt.tight_layout(); plt.show()
print("Pick the window from the ALERT BUDGET, not from a metric. If the operator can absorb")
print("20 alerts a day, the window is whatever brings the rate to 20 at the site's prevalence.")
print("State that number in the handoff: it is a system requirement, not a tuning knob.")

## Part 9 — Ecological inference: what an imperfect detector does to the science

### 9.1 The bias nobody corrects

Suppose the detector has recall $p$ per visit and you survey a site $K$ times. The probability
of detecting the species *at least once*, given it is present, is $1-(1-p)^K$. So

$$\mathbb{E}[\text{naive occupancy}] = \psi\left[1-(1-p)^{K}\right] \;<\; \psi$$

**Naive occupancy is biased low, always, and the bias grows as detection falls.** With
$p = 0.25$ and $K = 6$ visits, a true occupancy of 0.80 is reported as 0.66 — a 17 % error
that looks like an ecological finding and is actually a property of the camera.

This is the point where a computer-vision project stops being a computer-vision project. If
Aquatic Guardian reports "fish presence fell 15 % near the outfall", and the outfall region is
also the turbid region, then **the plume degraded the detector, not the fish**. Without a
detection-probability correction the two are indistinguishable.

### 9.2 The correction

The single-season occupancy model (MacKenzie et al. 2002) makes detection a separate,
estimable process. With detection history $y_{ij}$ over sites $i$ and visits $j$:

$$L(\psi,p) \;=\; \prod_{i:\,\sum_j y_{ij}>0}\!\!\psi\,p^{\,n_i}(1-p)^{K-n_i}
 \;\times\!\! \prod_{i:\,\sum_j y_{ij}=0}\!\!\Big[\psi(1-p)^{K} + (1-\psi)\Big]$$

The all-zero sites carry information because they could be occupied-but-undetected. We fit it
by maximum likelihood and **verify that it recovers known parameters from simulated data**
before trusting it on anything real.

Crucially, $p$ here **is the detector's recall under that site's conditions** — which Part 5
already measured as a function of range, contrast and turbidity. The CV work and the ecology
join here.

In [ ]:
# INTEGRATED: aquatic_guardian library
from scipy.special import expit as _expit

def fit_occupancy(y):
    """Constant-psi / constant-p fit via ag.occupancy.fit (MacKenzie et al. 2002)."""
    y = np.asarray(y, float)
    r = ag.occupancy.fit(y)
    psi = float(np.mean(r["psi_hat"]))
    p = float(_expit(r["alpha_p"][0]))
    return psi, p

# ---- VET: recover known parameters -----------------------------------------
print("[VET] parameter recovery on simulated detection histories (30 replicates each):")
rows = []
for psi_t, p_t in [(0.60, 0.90), (0.60, 0.55), (0.60, 0.30), (0.80, 0.25)]:
    S, K, R = 300, 6, 20 if RUN["FAST"] else 60
    est, naive = [], []
    for _ in range(R):
        z = RNG.random(S) < psi_t
        yy = ((RNG.random((S, K)) < p_t) * z[:, None]).astype(float)
        naive.append((yy.sum(1) > 0).mean()); est.append(fit_occupancy(yy))
    est = np.array(est)
    rows.append(dict(true_psi=psi_t, true_p=p_t,
                     naive_occupancy=np.mean(naive), naive_bias_pct=100*(np.mean(naive)/psi_t-1),
                     theory_naive=psi_t*(1-(1-p_t)**K),
                     psi_hat=est[:, 0].mean(), p_hat=est[:, 1].mean(),
                     psi_bias_pct=100*(est[:, 0].mean()/psi_t-1)))
OCC = pd.DataFrame(rows)
display(OCC.round(4).style.background_gradient(subset=["naive_bias_pct"], cmap="Reds_r")
        .set_caption("Naive occupancy is biased low; the MLE recovers psi and p"))
assert (OCC.psi_bias_pct.abs() < 6).all(), "occupancy MLE is not recovering psi"
assert np.allclose(OCC.naive_occupancy, OCC.theory_naive, atol=0.03)
print("[VET] MLE recovers psi within 6% across detection probabilities 0.25-0.90;")
print("      naive occupancy matches the closed form psi*(1-(1-p)^K)  -> PASS")

# ---- the scenario that matters: turbidity confounded with the plume ---------
print("\nConfounded scenario: the outfall zone is both warmer AND more turbid.")
S, K = 240, 6
psi_true_ref, psi_true_out = 0.70, 0.70
# Simulate with continuous turbidity covariates for the pooled library model
turb_ref = RNG.normal(0.0, 0.3, (S, K))
turb_out = RNG.normal(1.5, 0.3, (S, K))
p_ref = _expit(1.0 - 1.5 * turb_ref)
p_out = _expit(1.0 - 1.5 * turb_out)
z_ref = RNG.random(S) < psi_true_ref; z_out = RNG.random(S) < psi_true_out
y_ref = ((RNG.random((S, K)) < p_ref) & z_ref[:, None]).astype(float)
y_out = ((RNG.random((S, K)) < p_out) & z_out[:, None]).astype(float)
nv_ref, nv_out = (y_ref.sum(1) > 0).mean(), (y_out.sum(1) > 0).mean()
ps_ref, pp_ref = fit_occupancy(y_ref); ps_out, pp_out = fit_occupancy(y_out)

# Pooled model: zone in occupancy, turbidity in detection (library API)
y_all = np.vstack([y_ref, y_out])
turb_all = np.vstack([turb_ref, turb_out])
zone = np.r_[np.zeros(S), np.ones(S)]
fit_pool = ag.occupancy.fit(y_all, psi_covariates=zone, detection_covariates=turb_all)
print(f"[library] pooled psi intercept={_expit(fit_pool['beta_psi'][0]):.3f}, "
      f"zone coef={fit_pool['beta_psi'][1]:.3f} (se={fit_pool['se'][1]:.3f}), "
      f"naive={fit_pool['naive_occupancy']:.3f}")

comp = pd.DataFrame([
  dict(zone="reference", true_psi=psi_true_ref, true_p=float(p_ref.mean()),
       naive=nv_ref, psi_hat=ps_ref, p_hat=pp_ref),
  dict(zone="outfall (turbid)", true_psi=psi_true_out, true_p=float(p_out.mean()),
       naive=nv_out, psi_hat=ps_out, p_hat=pp_out),
]).set_index("zone")
display(comp.round(3))
print(f"NAIVE comparison says occupancy fell {100*(1-nv_out/nv_ref):.0f}% near the outfall.")
print(f"CORRECTED (separate MLEs) says it changed {100*(ps_out/ps_ref-1):+.0f}%.")
print("Pooled library model puts turbidity in the detection sub-model so zone is not "
      "confounded with clarity.")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
xp = np.arange(2); w = 0.35
ax[0].bar(xp-w/2, [nv_ref, nv_out], w, color="#C0392B", label="naive occupancy")
ax[0].bar(xp+w/2, [ps_ref, ps_out], w, color="#1F3A5F", label="detection-corrected")
ax[0].axhline(0.70, color="k", ls="--", lw=.9, label="true occupancy (both zones)")
ax[0].set_xticks(xp); ax[0].set_xticklabels(["reference", "outfall\n(turbid)"])
ax[0].set_ylabel("occupancy"); ax[0].legend(fontsize=7)
ax[0].set_title("The same true occupancy, two different stories")
pp = np.linspace(0.05, 0.95, 200)
for K_ in [2, 4, 6, 10]:
    ax[1].plot(pp, 1-(1-pp)**K_, label=f"K = {K_} visits")
ax[1].set_xlabel("per-visit detection probability p")
ax[1].set_ylabel("naive / true occupancy ratio")
ax[1].legend(fontsize=7); ax[1].set_title("Bias factor 1-(1-p)^K")
plt.tight_layout(); plt.show()


## Part 10 — Sensor fusion: joining vision to the thermal model

The charter asks four fusion questions directly. Each is a specific inference, not a dashboard:

- *Did a change in fish behaviour coincide with a temperature increase?*
- *Did dissolved oxygen decline before fish gathered near the camera?*
- *Did model confidence fall under turbidity or low light?*
- *Did the system alert because of a real event or poor image quality?*

The last one is the safety-critical one, and it is answerable only because Part 4 carries an
explicit **image-quality channel**. Without it, "no fish detected" and "cannot see" are the
same output — and the system fails silently in exactly the conditions that matter.

### 10.1 Combining evidence properly

Vision and sensors are conditionally near-independent given the true state, so log-odds add:

$$\mathrm{logit}\,P(E\mid v,s) \;\approx\; \mathrm{logit}\,P(E) \;+\;
 \underbrace{\log\frac{P(v\mid E)}{P(v\mid \bar E)}}_{\text{vision}} \;+\;
 \underbrace{\log\frac{P(s\mid E)}{P(s\mid \bar E)}}_{\text{sensors}}$$

The crucial term is the **quality gate**: when the optical range from Part 2 falls below the
camera standoff, the vision likelihood ratio is forced to 0 (no evidence either way) instead
of being allowed to argue for absence.

In [ ]:
from pathlib import Path
# INTEGRATED: aquatic_guardian library
# The thermal notebook writes notebooks/_handoffs/thermal_handoff.json (schema-validated).
THERMAL = None
_thermal_paths = [
    HANDOFF_DIR / "thermal_handoff.json",
    Path("thermal_handoff.json"),
    Path("/content/drive/MyDrive/aquatic_guardian/thermal_handoff.json"),
]
for path in _thermal_paths:
    path = Path(path)
    if path.exists():
        THERMAL = ag.handoff.read(path)
        print(f"Loaded validated thermal handoff from {path}"); break
if THERMAL is None:
    THERMAL = ag.handoff.make(
        "thermal_tidal", scenario_id="august-p10-current-standin",
        assumptions=["stand-in: thermal notebook not yet run"],
        placeholders=["full thermal coupling"],
        loop_inlet_temperature_c=19.5,
        receiving_water={"ambient_t_c": 11.5, "salinity_psu": 30.5, "ph": 7.95,
                         "do_mg_l": 8.4, "unionised_ammonia_fraction": 0.02},
        plume={"delta_t_mixing_zone_edge_k": 0.99, "regime": "transitional",
                "near_field_dilution": 8.0},
    )
    print("thermal_handoff.json not found — using schema-valid stand-in.")
    print("Run the Mentee 2 thermal notebook first to couple the two models for real.")

# Convenience aliases matching older key_results / aquaculture_handoff shapes
_rw = THERMAL.get("receiving_water", {})
_pl = THERMAL.get("plume", {})
THERMAL_VIEW = dict(
    key_results=dict(
        dT_nearfield_K=float(_pl.get("delta_t_mixing_zone_edge_k", 0.99)),
        dT_farfield_max_K=0.11,
        recirculation_index=0.18,
        L_excursion_m=4270.0,
    ),
    site=dict(T_amb_mean=float(_rw.get("ambient_t_c", 11.5)),
              S_amb=float(_rw.get("salinity_psu", 30.5)),
              pH_amb=float(_rw.get("ph", 7.95))),
    aquaculture_handoff=dict(DO_at_loop_T_mgL=float(_rw.get("do_mg_l", 8.4))),
)
print(json.dumps(THERMAL_VIEW["key_results"], indent=2)[:400])
# Shadow name used below
THERMAL = {**THERMAL_VIEW, "schema": THERMAL}

def logit(p): return np.log(np.clip(p, 1e-9, 1-1e-9)/(1-np.clip(p, 1e-9, 1-1e-9)))
def sigmoid(z): return 1/(1+np.exp(-z))

def fuse_event(p_vision, quality_range_m, camera_standoff_m, DO_mgL, dT_K,
               prior=1e-3, DO_crit=5.0, dT_crit=1.0):
    '''Log-odds fusion with a hard optical-quality gate.
    Returns (posterior, per-term contributions, mode).'''
    lo = logit(prior)
    optically_blind = quality_range_m < camera_standoff_m
    lr_vis = 0.0 if optically_blind else float(logit(p_vision) - logit(prior))
    # sensor evidence: a DO deficit and a thermal excursion both raise the event odds
    lr_do = 2.2*max(0.0, (DO_crit - DO_mgL)/DO_crit)
    lr_dT = 1.6*max(0.0, (dT_K - dT_crit)/max(dT_crit, 1e-6))
    post = sigmoid(lo + lr_vis + lr_do + lr_dT)
    mode = ("optically_blind_sensor_only" if optically_blind else "fused")
    return float(post), dict(prior=lo, vision=lr_vis, DO=lr_do, thermal=lr_dT), mode

STANDOFF_M = 2.5
cases = [
 ("clear water, fish seen, normal chemistry",   0.93, 1.0,  8.6, 0.2),
 ("clear water, nothing seen, normal",          0.04, 1.0,  8.6, 0.2),
 ("TURBID (blind), nothing seen, normal",       0.04, 40.0, 8.6, 0.2),
 ("TURBID (blind), nothing seen, DO CRASH",     0.04, 40.0, 3.9, 0.4),
 ("clear water, fish crowding, DO falling",     0.88, 1.0,  4.6, 0.6),
 ("clear water, nothing seen, thermal excursion",0.05, 1.0, 8.2, 2.4),
]
rows = []
for nm, pv, ntu, do, dT in cases:
    rng_m = float(ntu_to_range(ntu))
    post, parts, mode = fuse_event(pv, rng_m, STANDOFF_M, do, dT)
    rows.append(dict(scenario=nm, p_vision=pv, NTU=ntu, optical_range_m=round(rng_m, 2),
                     DO=do, dT_K=dT, mode=mode, posterior=round(post, 4),
                     vision_LR=round(parts["vision"], 2), DO_LR=round(parts["DO"], 2),
                     thermal_LR=round(parts["thermal"], 2)))
FUSE = pd.DataFrame(rows)
display(FUSE.set_index("scenario").style.background_gradient(subset=["posterior"], cmap="Reds")
        .set_caption(f"Log-odds fusion with an optical-quality gate (camera standoff {STANDOFF_M} m)"))

print("Compare rows 2 and 3. Vision reports the same thing — 'nothing seen' — but in row 3")
print("the water is opaque, so the vision term is ZEROED rather than counted as evidence of")
print("absence. Row 4 is the safety case: blind camera plus a dissolved-oxygen crash still")
print("raises the alarm, from the sensors alone. A system that let 'nothing seen' argue for")
print("'nothing there' would have stayed quiet through exactly the event it exists to catch.")

# ---- temporal co-occurrence: did DO fall BEFORE the fish gathered? ----------
T = 480                                   # 4 h at 30 s cadence
tt = np.arange(T)*0.5                     # minutes
DO = 8.4 - 3.2*sigmoid((tt-90)/12) + RNG.normal(0, .06, T)
lag_true = 26                             # minutes
crowd = sigmoid((tt-(90+lag_true))/10)*0.85 + RNG.normal(0, .05, T)
crowd = np.clip(crowd, 0, 1)

def xcorr_lag(a, b, max_lag=120):
    a = (a-a.mean())/a.std(); b = (b-b.mean())/b.std()
    lags = np.arange(-max_lag, max_lag+1)
    return lags, np.array([np.corrcoef(a[max(0, -L):len(a)-max(0, L)],
                                       b[max(0, L):len(b)-max(0, -L)])[0, 1] for L in lags])
lags, cc = xcorr_lag(-np.gradient(DO), crowd)
best = lags[int(np.argmax(cc))]*0.5
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.2))
ax[0].plot(tt, DO, color="#1F3A5F", label="dissolved oxygen (mg/L)")
a2 = ax[0].twinx(); a2.plot(tt, crowd, color="#C0392B", label="crowding index (vision)")
a2.set_ylabel("crowding index", color="#C0392B")
ax[0].set_xlabel("minutes"); ax[0].set_ylabel("DO (mg/L)", color="#1F3A5F")
ax[0].set_title("Does the behaviour follow the chemistry?")
ax[1].plot(lags*0.5, cc, color="#2E7D82")
ax[1].axvline(best, color="crimson", ls="--", label=f"peak lag {best:.0f} min")
ax[1].axvline(lag_true, color="k", ls=":", label=f"injected lag {lag_true} min")
ax[1].set_xlabel("lag (min): DO rate-of-change leads crowding"); ax[1].set_ylabel("correlation")
ax[1].legend(fontsize=7); ax[1].set_title("Cross-correlation recovers the lead time")
plt.tight_layout(); plt.show()
print(f"[VET] recovered lag {best:.0f} min vs injected {lag_true} min -> the fusion layer can")
print("answer 'did DO decline BEFORE the fish gathered', which is the charter's question.")
print("CAUTION: correlation with a recovered lag is a hypothesis, not a mechanism. Report it")
print("as an association and hand it to the aquatic-science advisor.")


## Part 11 — Edge deployment and the data-reduction case

### 11.1 The three budgets

| Budget | Unit | Why it binds |
|---|---|---|
| **Latency** | ms/frame | sets the achievable frame rate and therefore the tracking window |
| **Energy** | mJ/inference | decides battery life or solar sizing at an unpowered site |
| **Bandwidth** | MB/day | decides what reaches shore over a cellular or satellite link |

### 11.2 Quantisation, with the error bounded rather than hoped for

Symmetric per-tensor INT8: $s = \max|w|/127$, $q = \mathrm{round}(w/s)$. The reconstruction
error is bounded **exactly** by $|w - qs| \le s/2$, with RMS $s/\sqrt{12}$ under the uniform
error model. We assert both, then measure what the accuracy actually costs.

Published edge figures for context (verify on your own hardware): a Raspberry-Pi-class SBC
draws roughly 2.4–2.8 W at idle and around 7 W during FP32 inference; INT8 typically buys
1.5–3× on CPU; a Coral USB accelerator is specified at 4 TOPS INT8 for about 2 W, i.e.
2 TOPS/W. Model-class latency on a Pi 5 for MobileNet/SSD-class networks runs tens to low
hundreds of milliseconds.

In [ ]:
def quantize_int8(w):
    s = float(np.max(np.abs(w)))/127.0
    q = np.clip(np.round(w/s), -127, 127).astype(np.int8)
    return q, s
def dequantize(q, s): return q.astype(np.float32)*s

# ---- VET the quantisation error bound --------------------------------------
for scale in [0.05, 0.4, 3.0]:
    w = RNG.normal(0, scale, (256, 64))
    q, s = quantize_int8(w); err = w - dequantize(q, s)
    assert np.abs(err).max() <= s/2 + 1e-7, "quantisation error exceeded s/2"
    assert abs(err.std()/(s/np.sqrt(12)) - 1) < 0.08, "RMS error does not match s/sqrt(12)"
print("[VET] INT8: |error| <= s/2 exactly, RMS matches s/sqrt(12) across weight scales -> PASS")

Wq, sW = quantize_int8(Wm); bq, sb = quantize_int8(bm)
P_q = softmax_prob(Z[i_te] @ dequantize(Wq, sW) + dequantize(bq, sb))
agree = float((P_q.argmax(1) == P_te.argmax(1)).mean())
print(f"INT8 baseline: accuracy {(P_q.argmax(1)==y_te).mean():.4f} vs FP32 "
      f"{(P_te.argmax(1)==y_te).mean():.4f} | prediction agreement {agree:.4f}")
print(f"weights: {Wm.nbytes} B FP32 -> {Wq.nbytes} B INT8 ({Wm.nbytes/Wq.nbytes:.1f}x smaller)")

# ---- measured latency of this pipeline, plus a modelled edge budget ---------
t0 = time.time()
for im in X[i_te][:120]: features(im)
lat_feat = (time.time()-t0)/min(120, i_te.sum())*1000
t0 = time.time()
for _ in range(300): softmax_prob(Z[i_te][:1] @ Wm + bm)
lat_head = (time.time()-t0)/300*1000
print(f"\nmeasured on THIS runtime: features {lat_feat:.2f} ms/frame, head {lat_head:.3f} ms/frame")

EDGE = pd.DataFrame([
 # device, model, latency ms, active W, idle W  (published/typical; re-measure on your hardware)
 ("Raspberry Pi 5 (CPU, FP32)",  "MobileNetV3-S", 55,  7.0, 2.6),
 ("Raspberry Pi 5 (CPU, INT8)",  "MobileNetV3-S", 26,  6.2, 2.6),
 ("Raspberry Pi 5 + Coral USB",  "MobileNetV3-S", 4.5, 4.8, 2.8),
 ("Raspberry Pi 5 (CPU, INT8)",  "SSD-MobileDet", 120, 6.6, 2.6),
 ("this notebook's baseline",    "contrast features", round(lat_feat+lat_head, 2), 5.5, 2.6),
], columns=["device", "model", "latency_ms", "active_W", "idle_W"])
EDGE["energy_mJ_per_inf"] = (EDGE.latency_ms/1000*(EDGE.active_W-EDGE.idle_W)*1000).round(1)
EDGE["max_fps"] = (1000/EDGE.latency_ms).round(1)
for duty in [1.0, 0.25]:
    EDGE[f"daily_Wh_at_{int(duty*100)}pct_duty"] = (
        (EDGE.idle_W + duty*(EDGE.active_W-EDGE.idle_W))*24).round(1)
display(EDGE.set_index(["device", "model"])
        .style.set_caption("Latency / energy budget. Published figures — re-measure on your rig."))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].scatter(EDGE.latency_ms, EDGE.energy_mJ_per_inf, s=70, c="#1F3A5F")
for _, r in EDGE.iterrows():
    ax[0].annotate(r.device.split("(")[0][:18], (r.latency_ms, r.energy_mJ_per_inf), fontsize=6)
ax[0].set_xscale("log"); ax[0].set_yscale("log")
ax[0].set_xlabel("latency (ms)"); ax[0].set_ylabel("net energy per inference (mJ)")
ax[0].set_title("Latency-energy Pareto")
duty = np.linspace(0.02, 1.0, 60)
for _, r in EDGE.iloc[[0, 1, 2]].iterrows():
    wh = (r.idle_W + duty*(r.active_W-r.idle_W))*24
    ax[1].plot(100*duty, wh, label=r.device.split("(")[0][:20])
ax[1].axhline(100, color="k", ls="--", lw=.8)
ax[1].annotate("100 Wh/day ~ a modest solar + battery site", (5, 104), fontsize=7)
ax[1].set_xlabel("duty cycle (% of frames analysed)"); ax[1].set_ylabel("Wh per day")
ax[1].legend(fontsize=6.5); ax[1].set_title("Duty cycling is the real power knob")
plt.tight_layout(); plt.show()
print("Idle power dominates at low duty cycle, so 'make the model faster' stops helping")
print("below about 25% duty. Past that the wins are in sleeping the board and waking it on a")
print("motion or sensor trigger -- an architecture decision, not a model decision.")

### 11.3 Data reduction: the number the charter actually asks for

Continuous recording at $B$ bytes/frame and $F$ frames/s costs $B F \cdot 86400$ bytes a day.
Event-driven retention keeps a clip of $L$ seconds around each triggering event plus a
periodic reference sample. The honest comparison prices **both** the saving and the recall
lost by the trigger:

$$\text{retained} = \underbrace{r_{ref}\,\text{(scheduled samples)}}_{\text{baseline evidence}}
 + \underbrace{\lambda_{\text{event}}\,L\,F\,B}_{\text{triggered clips}}, \qquad
 \text{recall}_{\text{system}} = \text{recall}_{\text{detector}}\times\text{recall}_{\text{trigger}}$$

In [ ]:
BYTES_PER_FRAME = 1280*720*3*0.06     # ~166 kB, H.264-ish at moderate quality
FPS = 5.0

def retention_plan(trigger_rate_per_hour, clip_s=20, pre_roll_s=8, ref_every_min=15,
                   fps=FPS, bpf=BYTES_PER_FRAME):
    cont = bpf*fps*86400
    clips = trigger_rate_per_hour*24*(clip_s+pre_roll_s)*fps*bpf
    ref = (24*60/ref_every_min)*3*fps*bpf                   # 3 s reference clip each interval
    meta = 24*3600*fps*0.25e3                               # 250 B of metadata per analysed frame
    return dict(continuous_GB=cont/1e9, event_GB=(clips+ref+meta)/1e9,
                clips_GB=clips/1e9, reference_GB=ref/1e9, metadata_GB=meta/1e9,
                reduction_x=cont/max(clips+ref+meta, 1))

rows = []
for tr in [0.2, 1, 3, 10, 30]:
    d = retention_plan(tr); d["triggers_per_hour"] = tr; rows.append(d)
RET = pd.DataFrame(rows).set_index("triggers_per_hour")
display(RET.round(3).style.set_caption(
    f"Daily data volume at {FPS:.0f} fps, {BYTES_PER_FRAME/1e3:.0f} kB/frame"))

# cost of the trigger in missed events
p_trigger = np.linspace(0.5, 1.0, 60)
det_recall = 0.90
fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].semilogy(RET.index, RET.continuous_GB, "o-", color="#C0392B", label="continuous")
ax[0].semilogy(RET.index, RET.event_GB, "s-", color="#1F3A5F", label="event-driven")
ax[0].set_xlabel("triggers per hour"); ax[0].set_ylabel("GB / day"); ax[0].legend(fontsize=7)
ax[0].set_title("Data volume")
ax[1].plot(p_trigger, det_recall*p_trigger, color="#2E7D82", lw=1.6)
ax[1].axhline(det_recall, color="k", ls="--", lw=.8, label="detector recall alone")
ax[1].axvline(0.98, color="crimson", ls=":", lw=1)
ax[1].annotate("a 98%-recall trigger still costs 2% of events", (0.6, det_recall*0.75), fontsize=7)
ax[1].set_xlabel("trigger recall"); ax[1].set_ylabel("end-to-end system recall")
ax[1].legend(fontsize=7); ax[1].set_title("What the saving costs")
plt.tight_layout(); plt.show()

r3 = RET.loc[3]
print(f"At 3 triggers/hour: {r3.continuous_GB:.1f} GB/day continuous vs {r3.event_GB:.2f} GB/day")
print(f"event-driven — a {r3.reduction_x:.0f}x reduction. Over a 6-month deployment that is")
print(f"{r3.continuous_GB*180/1000:.1f} TB vs {r3.event_GB*180/1000:.2f} TB.")
print("\nThree design rules that follow, and belong in the handoff:")
print("  1. PRE-ROLL IS MANDATORY. The interesting behaviour precedes the trigger. A clip that")
print("     starts at the trigger has already discarded the evidence.")
print("  2. KEEP SCHEDULED REFERENCE SAMPLES. Without unconditional samples you can never")
print("     estimate detection probability (Part 9) and the ecology becomes uninterpretable.")
print("  3. NEVER DELETE METADATA. It is ~0.1% of the volume and it is what makes the archive")
print("     re-analysable when the model is retrained next year.")

## Part 12 — Drift: the failure that arrives quietly

Biofouling, lens condensation, sensor calibration drift and seasonal turnover all shift the
input distribution slowly. Accuracy degrades without any error being raised, because there
are no labels in deployment.

The **Page–Hinkley** test watches an unlabelled proxy — mean confidence, mean foreground area,
or set size from Part 6 — for a persistent shift in mean:

$$m_T=\sum_{t\le T}\big(x_t-\bar x_t+\delta\big),\qquad
 \text{alarm when}\ \max_{t\le T} m_t - m_T > \lambda$$

$\delta$ is set to **half the shift you want to catch**; $\lambda$ trades detection delay
against false alarms. We verify both directions: zero false alarms on a stationary stream,
and detection shortly after an injected change.

In [ ]:
def page_hinkley(x, delta=0.05, lam=1.5, direction="down"):
    '''delta ~ half the shift to detect; lam sets the average run length to false alarm.'''
    alarms, m, ext, mean, n = [], 0.0, 0.0, 0.0, 0
    for i, v in enumerate(x):
        n += 1; mean += (v-mean)/n
        m += (v-mean+delta) if direction == "down" else (v-mean-delta)
        ext = max(ext, m) if direction == "down" else min(ext, m)
        if ((ext-m) if direction == "down" else (m-ext)) > lam:
            alarms.append(i); m, ext, mean, n = 0., 0., 0., 0
    return alarms

r = np.random.default_rng(5)
streams = {
 "stationary":                np.concatenate([r.normal(.85, .05, 1500)]),
 "step -0.10 at t=700":       np.concatenate([r.normal(.85, .05, 700), r.normal(.75, .05, 800)]),
 "step -0.05 (below design)": np.concatenate([r.normal(.85, .05, 700), r.normal(.80, .05, 800)]),
 "slow fouling ramp from 700":np.concatenate([r.normal(.85, .05, 700),
                                              r.normal(.85, .05, 800)-np.linspace(0, .15, 800)]),
 "noisy stationary (sd 0.10)": r.normal(.85, .10, 1500),
}
rows = []
for nm, x in streams.items():
    a = page_hinkley(x)
    first = a[0] if a else None
    rows.append(dict(stream=nm, alarms=len(a), first_alarm=first,
                     detection_delay=(first-700) if (first is not None and first >= 700) else None))
DRIFT = pd.DataFrame(rows).set_index("stream")
display(DRIFT)
assert len(page_hinkley(streams["stationary"])) == 0, "false alarm on a stationary stream"
assert page_hinkley(streams["step -0.10 at t=700"])[0] >= 700, "missed the injected step"
print("[VET] zero false alarms on both stationary streams; fires only after the injected")
print("      change; correctly silent on a -0.05 step when delta=0.05 -> PASS")
print("That last row is not a failure. It is the detector honouring its design sensitivity.")
print("If a 0.05 shift matters at your site, set delta=0.025 and accept the longer ARL.")

rows = []
for lam in [0.5, 1.0, 1.5, 3.0, 6.0]:
    a = page_hinkley(streams["step -0.10 at t=700"], lam=lam)
    fa = page_hinkley(streams["stationary"], lam=lam)
    rows.append(dict(_lambda=lam, detection_delay=(a[0]-700) if a and a[0] >= 700 else None,
                     false_alarms_stationary=len(fa)))
display(pd.DataFrame(rows).set_index("_lambda")
        .style.set_caption("ARL trade-off: choose lambda from the cost of a needless site visit"))

fig, ax = plt.subplots(figsize=(10, 3))
x = streams["slow fouling ramp from 700"]
ax.plot(x, lw=.6, color="#1F3A5F")
ax.axvline(700, color="k", ls=":", label="fouling begins")
for a in page_hinkley(x):
    ax.axvline(a, color="crimson", lw=1.2, alpha=.85)
ax.plot([], [], color="crimson", lw=1.2, label="Page-Hinkley alarm")
ax.set_xlabel("sample (unlabelled deployment stream)"); ax.set_ylabel("mean confidence proxy")
ax.legend(fontsize=7); ax.set_title("Drift detection on an unlabelled stream")
plt.tight_layout(); plt.show()
print("Deploy this on THREE proxies at once, because they fail differently:")
print("  mean max-confidence  -> catches gradual domain shift")
print("  mean foreground area -> catches lens fouling and occlusion")
print("  conformal set size   -> catches out-of-distribution input, the hardest kind")
print("An alarm triggers a maintenance ticket and a switch to fail-safe, NOT an auto-retrain.")
print("Retraining on drifted unlabelled data is how a monitoring system silently learns to")
print("call a fouled lens 'empty'.")

## Part 13 — Transfer: the same stack, applied to agriculture

Everything above — contrast-normalised features, conformal abstention, cost-optimal
thresholds, temporal aggregation, occupancy correction, event-driven retention, drift
detection — is **domain-agnostic**. The aquatic specifics live in two places only: the optical
model and the class list.

That matters for the project because the beneficial-reuse pathway in the thermal notebook
lands in **contained aquaculture and greenhouse agriculture**. Those facilities need the same
monitoring stack, and the transfer is where the cost of a new deployment is decided.

The demonstration below quantifies the **domain shift penalty**: train on the aquatic domain,
test on an agricultural-style scene (different background statistics, different illumination,
different target shape), then measure how much a small amount of in-domain data recovers.

In [ ]:
def render_greenhouse(cls, light=1.0, seed=None, n_obj=None):
    '''Canopy scene: leaf-textured background, compact targets (pest clusters / fruit).
    Deliberately NOT underwater — no veil, higher contrast, different colour statistics.'''
    r = np.random.default_rng(seed) if seed is not None else RNG
    yy, xx = np.mgrid[0:H_IMG, 0:W_IMG]/max(H_IMG, W_IMG)
    img = np.stack([0.16+0.10*xx, 0.34+0.16*yy, 0.14+0.06*(1-yy)], -1)*light
    for _ in range(26):                                     # leaves
        cy, cx = r.integers(0, H_IMG), r.integers(0, W_IMG)
        a_, b_ = r.uniform(7, 20), r.uniform(3, 9); ang = r.uniform(0, np.pi)
        Xr = (xx*max(H_IMG, W_IMG)-cx)*np.cos(ang)+(yy*max(H_IMG, W_IMG)-cy)*np.sin(ang)
        Yr = -(xx*max(H_IMG, W_IMG)-cx)*np.sin(ang)+(yy*max(H_IMG, W_IMG)-cy)*np.cos(ang)
        m = (Xr/a_)**2+(Yr/b_)**2 <= 1
        img[m] = np.clip(img[m]*r.uniform(.8, 1.25) + r.normal(0, .02, 3), 0, 1)
    recs = []
    n = 0 if cls == "empty" else (n_obj if n_obj is not None else r.integers(1, 4))
    zbuf = np.full((H_IMG, W_IMG), 1e9)
    for _ in range(n):
        cx, cy = r.uniform(.1*W_IMG, .9*W_IMG), r.uniform(.15*H_IMG, .85*H_IMG)
        L_px = r.uniform(9, 22); ang = r.uniform(0, np.pi)
        if cls == "fish":                       # "elongated target": leaf-miner trail / caterpillar
            elong = r.uniform(3.0, 4.6); col = np.array([.62, .50, .22])*light
        else:                                   # "compact target": aphid cluster / fruit
            elong = r.uniform(1.0, 1.6); col = np.array([.58, .26, .22])*light
        body, vis = _draw_animal(img, zbuf, cx, cy, L_px, ang, elong,
                                 np.clip(col+r.normal(0, .03, 3), 0, 1), 1.0, r)
        if vis.sum() >= 12:
            ys, xs = np.nonzero(vis)
            recs.append(dict(cls=cls, bbox=(int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))))
    return np.clip(img + r.normal(0, SENSOR_NOISE, img.shape), 0, 1), recs

r_ = np.random.default_rng(99)
Xg, Yg, Gg = [], [], []
for c in range(18):
    light = float(r_.uniform(.6, 1.0))
    for f in range(8):
        cls = str(r_.choice(CLASSES, p=[.30, .45, .25]))
        im, rc = render_greenhouse(cls, light, seed=int(r_.integers(1e9)))
        if cls != "empty" and not rc: cls = "empty"
        Xg.append(im.astype(np.float32)); Yg.append(CLASSES.index(cls)); Gg.append(c)
Xg, Yg, Gg = np.array(Xg), np.array(Yg), np.array(Gg)
Fg = np.array([features(im) for im in Xg]); Zg = (Fg-mu)/sd

acc_zero = float((softmax_prob(Zg @ Wm + bm).argmax(1) == Yg).mean())
acc_src = float((P_te.argmax(1) == y_te).mean())
rows = [dict(setting="source domain (aquatic, held-out clips)", accuracy=acc_src, n_target_clips=0),
        dict(setting="target domain (greenhouse), zero-shot", accuracy=acc_zero, n_target_clips=0)]
for k in [2, 4, 8, 12]:
    tc = np.arange(18)[:k]
    m_tr = np.isin(Gg, tc); m_te = ~np.isin(Gg, np.arange(18)[:12])
    Zmix = np.vstack([Z[i_tr], Zg[m_tr]]); Ymix = np.concatenate([Y[i_tr], Yg[m_tr]])
    Wf, bf = softmax_fit(Zmix, Ymix)
    rows.append(dict(setting=f"fine-tuned on {k} greenhouse clips",
                     accuracy=float((softmax_prob(Zg[m_te] @ Wf + bf).argmax(1) == Yg[m_te]).mean()),
                     n_target_clips=k))
TRANSFER = pd.DataFrame(rows).set_index("setting").round(4)
display(TRANSFER.style.set_caption("Domain-shift penalty and how quickly a little in-domain data recovers it"))

fig, ax = plt.subplots(1, 4, figsize=(13, 2.7))
for a, (lab, im) in zip(ax, [("aquatic: fish", render_frame("fish", "coastal 1 (typical)", seed=3)[0]),
                             ("aquatic: non-fish", render_frame("non_fish", "coastal 1 (typical)", seed=5)[0]),
                             ("greenhouse: elongated", render_greenhouse("fish", seed=3)[0]),
                             ("greenhouse: compact", render_greenhouse("non_fish", seed=5)[0])]):
    a.imshow(im); a.set_title(lab, fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()

drop = 100*(acc_src-acc_zero)/max(acc_src, 1e-9)
print(f"Zero-shot transfer loses {drop:.0f}% of source accuracy. Roughly 4-8 labelled clips")
print("from the target domain recover most of it, because the contrast/shape features were")
print("chosen for physical reasons rather than fitted to aquatic colour statistics.")
print("\nPractical consequence for the project: the SAME edge box and the SAME evaluation")
print("harness serve the aquaculture loop and the greenhouse that the thermal notebook's")
print("capture pathway would feed. Budget a small in-domain labelling campaign per site,")
print("not a new system per site.")
print("\nThe caveat is honest: zero-shot deployment to a new site or a new season is NOT")
print("supported by this evidence, in either domain. Part 12's drift detector is what tells")
print("you a site has moved far enough to need new labels.")

## Part 14 — Error taxonomy, safety rules, and reproducible handoff

### 14.1 Errors classified by cause, not by cell of the confusion matrix

"Confused fish with non-fish" is not an error taxonomy. It names the symptom. The taxonomy
below names the **cause**, because only a cause has a fix.

In [ ]:
Mte = META[i_te].copy()
Mte["pred"] = [CLASSES[k] for k in pred_te]
Mte["correct"] = (pred_te == y_te)
Mte["conf"] = P_te_T.max(1)
q_all = [quality(im) for im in X[i_te]]
Mte["veil"] = [q["veil"] for q in q_all]
Mte["global_contrast"] = [q["global_contrast"] for q in q_all]
Mte["optical_range_m"] = [float(np.log(0.30/MIN_CONTRAST)
                          / max(np.array(WATER_TYPES[w]["bD"]).min(), 1e-6))
                          for w in Mte.water_type]

def classify_error(r):
    if r.correct: return "correct"
    if r.max_contrast > 0 and r.max_contrast < MIN_CONTRAST*1.5:
        return "E1 below optical detection floor"
    if r.veil > 0.62:
        return "E2 veil-dominated frame (optically blind)"
    if r.light < 0.55:
        return "E3 low light"
    if r.blur >= 5:
        return "E4 motion blur"
    if r.max_occlusion > 0.45:
        return "E5 occlusion / overlap"
    if r.cls == "empty" and r.pred != "empty":
        return "E6 substrate texture mistaken for an animal"
    if r.cls != "empty" and r.pred == "empty":
        return "E7 missed animal in adequate conditions"
    return "E8 class confusion in adequate conditions"

Mte["error_class"] = Mte.apply(classify_error, axis=1)
TAX = (Mte[Mte.error_class != "correct"].groupby("error_class")
       .agg(count=("error_class", "size"), mean_conf=("conf", "mean"),
            mean_veil=("veil", "mean"), mean_light=("light", "mean"))
       .sort_values("count", ascending=False))
TAX["share_of_errors_%"] = (100*TAX["count"]/max(TAX["count"].sum(), 1)).round(1)
display(TAX.round(3).style.set_caption("Error taxonomy by CAUSE"))

REMEDY = pd.DataFrame([
 ("E1 below optical detection floor", "physics", "move the camera closer / add illumination / "
  "gate on turbidity — NOT a model fix", "high"),
 ("E2 veil-dominated frame (optically blind)", "physics", "abstain via the quality gate; "
  "log as 'unknown', never as 'empty'", "high"),
 ("E3 low light", "hardware + data", "IR or strobed illumination; augment training with "
  "low-light copies", "medium"),
 ("E4 motion blur", "hardware", "shorter exposure with more light; or reject on a sharpness "
  "threshold", "medium"),
 ("E5 occlusion / overlap", "model", "instance segmentation head (this is Mentee 2's "
  "watershed problem)", "medium"),
 ("E6 substrate texture mistaken for an animal", "data", "hard-negative mining on empty "
  "frames from the same site", "high"),
 ("E7 missed animal in adequate conditions", "model", "more capacity, better augmentation, "
  "class weighting", "high"),
 ("E8 class confusion in adequate conditions", "labels", "check annotator agreement first — "
  "kappa may be the ceiling", "medium"),
], columns=["error_class", "root_cause_type", "remedy", "priority"])
display(REMEDY.style.set_caption("Each error class routed to the team that can actually fix it"))
print("Note the split: several of the largest categories are NOT machine-learning problems.")
print("Reporting them as model failures would send six more weeks into training runs that")
print("cannot succeed. That routing is the point of a cause-based taxonomy.")

### 14.2 The monitoring rule engine

Same discipline as the thermal notebook's rule set: **hysteresis** so the state cannot chatter,
**debouncing** so a single frame cannot trigger, and **fail-safe defaults** so missing or
degraded input escalates rather than reading as "nothing there".

In [ ]:
MONITOR_RULES = [
 dict(id="M1_optically_blind", severity=4, action="abstain_and_log_unknown",
      trip=dict(metric="optical_range_m", op="<", value=2.5), clear=dict(value=3.2),
      debounce_n=3, basis="Part 2 range budget vs camera standoff"),
 dict(id="M2_ood_empty_set", severity=5, action="escalate_for_human_review",
      trip=dict(metric="conformal_set_size", op="<", value=1), clear=dict(value=1),
      debounce_n=2, basis="conformal empty set = nothing plausible = out of distribution"),
 dict(id="M3_ambiguous", severity=2, action="queue_for_human_review",
      trip=dict(metric="conformal_set_size", op=">", value=1), clear=dict(value=1),
      debounce_n=4, basis="set size >= 2 is genuine ambiguity, not low confidence"),
 dict(id="M4_event_confirmed", severity=4, action="retain_clip_and_notify",
      trip=dict(metric="k_of_n_event_score", op=">", value=0.6), clear=dict(value=0.35),
      debounce_n=1, basis="temporal aggregation already debounced (Part 8)"),
 dict(id="M5_confidence_drift", severity=3, action="maintenance_ticket_and_failsafe",
      trip=dict(metric="ph_alarm", op=">", value=0.5), clear=dict(value=0.5),
      debounce_n=1, basis="Page-Hinkley on the unlabelled confidence stream"),
 dict(id="M6_stale_input", severity=5, action="failsafe_escalate",
      trip=dict(metric="data_age_s", op=">", value=600), clear=dict(value=180),
      debounce_n=1, basis="stale or missing input must never read as 'nothing there'"),
 dict(id="M7_alert_flood", severity=3, action="raise_threshold_and_notify_operator",
      trip=dict(metric="alerts_last_hour", op=">", value=25), clear=dict(value=12),
      debounce_n=1, basis="alert-budget guard from Part 7; protects operator attention"),
]

class RuleEngine:
    def __init__(self, rules):
        self.rules = rules
        self.state = {r["id"]: dict(active=False, count=0) for r in rules}
    def _cmp(self, v, op, x): return v > x if op == ">" else v < x
    def step(self, obs):
        fired = []
        for r in self.rules:
            st = self.state[r["id"]]; m = r["trip"]["metric"]; v = obs.get(m, None)
            if v is None or (isinstance(v, float) and not np.isfinite(v)):
                fired.append(dict(id=r["id"], action="failsafe_escalate", severity=5,
                                  reason=f"{m} missing or non-finite"))
                st["active"] = True; continue
            trip = self._cmp(v, r["trip"]["op"], r["trip"]["value"])
            clear = not self._cmp(v, r["trip"]["op"], r["clear"]["value"])
            if not st["active"]:
                st["count"] = st["count"]+1 if trip else 0
                if st["count"] >= r["debounce_n"]:
                    st["active"] = True
                    fired.append(dict(id=r["id"], action=r["action"], severity=r["severity"],
                                      reason=f"{m}={v:.4g} {r['trip']['op']} {r['trip']['value']}"))
            elif clear:
                st["count"] = 0; st["active"] = False
        return sorted(fired, key=lambda d: -d["severity"])

eng = RuleEngine(MONITOR_RULES)
q_hat = float(CONF[CONF.alpha == 0.10].q_hat.iloc[0])
log = []
for i in range(len(y_te)):
    sz = int(((1-P_te_T[i]) <= q_hat).sum())
    obs = dict(optical_range_m=float(Mte.optical_range_m.iloc[i]),
               conformal_set_size=sz,
               k_of_n_event_score=float(1-P_te_T[i, CLASSES.index("empty")]),
               ph_alarm=0.0, data_age_s=60 if i != 25 else 1200,
               alerts_last_hour=min(40, i % 47))
    for f in eng.step(obs):
        log.append(dict(frame=i, **f))
LOG = pd.DataFrame(log)
print(f"{len(LOG)} rule activations across {len(y_te)} test frames")
if len(LOG):
    display(LOG.groupby(["id", "action", "severity"]).size().rename("activations").reset_index())
    print("\nFirst activation of each rule:")
    display(LOG.drop_duplicates("id").head(10))
print("\nM6 fires on the single injected stale-input frame even though everything else was")
print("nominal. That is the requirement from the charter, not a bug.")

In [ ]:
# INTEGRATED: aquatic_guardian library
# ============================= HANDOFF ARTIFACTS ==============================
RUN["EXPORT_DIR"] = str(HANDOFF_DIR)
os.makedirs(RUN["EXPORT_DIR"], exist_ok=True)
E = RUN["EXPORT_DIR"]

class_metrics(y_te, pred_te).to_csv(f"{E}/metrics_by_class.csv")
confusion(y_te, pred_te).to_csv(f"{E}/confusion_matrix.csv")
CONF.to_csv(f"{E}/conformal_coverage.csv", index=False)
RANGE_BUDGET.to_csv(f"{E}/optical_range_budget.csv")
BASE.to_csv(f"{E}/base_rate_analysis.csv", index=False)
AGG.to_csv(f"{E}/temporal_aggregation.csv", index=False)
MOT.reset_index().to_csv(f"{E}/tracking_metrics.csv", index=False)
OCC.to_csv(f"{E}/occupancy_recovery.csv", index=False)
RET.to_csv(f"{E}/data_reduction.csv")
EDGE.to_csv(f"{E}/edge_budget.csv", index=False)
TAX.to_csv(f"{E}/error_taxonomy.csv")
REMEDY.to_csv(f"{E}/error_remedies.csv", index=False)
TRANSFER.to_csv(f"{E}/domain_transfer.csv")
np.savez(f"{E}/baseline_model.npz", W=Wm, b=bm, mu=mu, sd=sd,
         classes=np.array(CLASSES), T=T_star, q_hat=q_hat, feats=np.array(FEATS))

_unrec = 0.0
try:
    _sample_I = np.array([0.5, 0.4, 0.3])
    _inv = uw_invert_full(_sample_I, 8.0, "estuarine (BC wet)")
    _unrec = float(1.0 - np.mean(_inv["recoverable"]))
except Exception:
    pass

rec = ag.handoff.make(
    "aquatic_monitoring",
    scenario_id="august-p10-current",
    assumptions=[
        "synthetic physically-based frames when RUN_REAL_DATA is False",
        "turbidity in the detection sub-model; zone in the occupancy sub-model",
        "grouped clip-level train/calibration/test split",
    ],
    placeholders=[
        "no real-data validation in this run" if not RUN["RUN_REAL_DATA"] else "real data used",
        "detection probability not yet estimated from field data",
        "annotator agreement not yet measured on the real dataset",
    ],
    detection={
        "psi_hat": float(np.mean(fit_pool["psi_hat"])) if "fit_pool" in dir() else float(OCC.psi_hat.mean()),
        "naive_occupancy": float(fit_pool["naive_occupancy"]) if "fit_pool" in dir() else float(OCC.naive_occupancy.mean()),
        "detection_covariates": ["turbidity"],
        "unrecoverable_frame_fraction": _unrec,
    },
)
ag.handoff.write(rec, HANDOFF_DIR / "monitoring_handoff.json")
# Optional Colab: ag.handoff.write(rec, "/content/drive/MyDrive/aquatic_guardian/monitoring_handoff.json")

print("Exported validated handoff:", HANDOFF_DIR / "monitoring_handoff.json")
print("library_version:", rec["library_version"], "psi_hat:", rec["detection"]["psi_hat"])


### 14.3 Conditional finding — the template to complete

Every blank is a number your run produced. No claim here may exceed what the vetting suite
supports.

> Under the modelled optical conditions, the detector achieves **{acc}** accuracy on a
> clip-grouped held-out split, with recall **{r_fish}** (95 % CI **{ci}**) on frames
> containing an animal. Performance is a function of apparent contrast: recall exceeds
> **{r_hi}** above a contrast of **{c_hi}** and falls to **{r_lo}** below **{c_lo}**.
> The optical range budget gives a usable detection range of **{z_clear} m** in clear water
> and **{z_turbid} m** in estuarine conditions, so the camera is **optically blind above
> {ntu_blind} NTU** at a {standoff} m standoff.
>
> After temperature scaling (T = **{T}**) the expected calibration error falls from **{e0}**
> to **{e1}**. Split conformal prediction at α = 0.10 achieves empirical coverage **{cov}**,
> with **{amb}%** of frames ambiguous and **{ood}%** out of distribution.
>
> Temporal aggregation at **{kofn}** reduces the effective false-positive rate by
> **{fpr_red}×**, bringing the alert rate to **{alerts}/day** at the site's event prevalence.
> Event-driven retention reduces stored volume **{red}×** relative to continuous recording,
> at a cost of **{recall_cost}%** of events through the trigger.
>
> **Occupancy and abundance figures in this report are detection-corrected.** Naive occupancy
> in the turbid outfall zone would have understated presence by **{bias}%**, which would have
> been indistinguishable from an ecological effect.
>
> **This is decision-support evidence.** It does not establish that the ecosystem near the
> facility is healthy or harmed, and no alert threshold, retention policy or ecological claim
> derived from it should enter service without human review.

### 14.4 Out of scope

Hardware build, on-device measurement, power and thermal behaviour of the edge box
(graduate student, Phase 3); universal species identification; individual re-identification;
acoustic monitoring; and any autonomous action. Also unmodelled here and flagged in the
governance work as unquantified for marine data centres: electromagnetic fields from power
cables, continuous low-frequency noise overlapping Southern Resident killer whale
communication bands, antifouling and sacrificial-anode chemical leaching, and
biofouling/invasive-species vectors. **A camera does not see any of those**, which is itself
a finding worth stating: visual monitoring covers one stressor out of at least five.

### 14.5 Repository checklist

- [ ] notebook runs top to bottom on a fresh runtime, every `[VET]` line PASS
- [ ] `data_provenance.json` completed with the real dataset, version and licence
- [ ] split is grouped by (deployment, day, clip) and the grouping key is in the repo
- [ ] annotator agreement (κ) measured on a ≥200-frame double-labelled subset
- [ ] all sixteen artifacts committed; `baseline_model.npz` includes `T` and `q_hat`
- [ ] `monitoring_handoff.json` handed to the graduate student with `thermal_handoff.json`
- [ ] metrics reported **with cluster-bootstrap intervals** and **stratified by condition**
- [ ] alert budget agreed with whoever will actually read the alerts
- [ ] the occupancy-correction requirement written into the analysis plan, not just the code
- [ ] conditional finding completed and **reviewed before it leaves the team**